# ShiftGuard10 — Merged Notebook (v1 → v4)

This notebook consolidates all four training versions for the **ShiftGuard10 Robust Image Classification** challenge.
Each version builds on the previous, progressively improving the public leaderboard score.

| Version | Key Change | Best Val F1 |
|---------|-----------|-------------|
| **v1** | WideResNet-28-10 + SWA baseline (200 epochs, 10-pass TTA) | 0.9106 |
| **v2** | AugMix + JSD consistency loss (20-pass TTA) | 0.9231 |
| **v3** | Pseudo-labeling + corruption augmentations (30-pass TTA) | 0.9262 |
| **v4** | Iterative pseudo-labeling iter 2 (50-pass TTA) | **0.9556** |

---

## ShiftGuard10 — v1: WideResNet-28-10 + SWA (Baseline)

_WideResNet-28-10 | RandAugment | 200 Epochs | 10-pass TTA | **Best Val F1: 0.9106**_

---

In [1]:
# =============================================================================
# ShiftGuard10 — Final Solution  |  Single Fold, 200 Epochs, T4 x2
# =============================================================================
# Philosophy: keep everything that PROVED to work at 0.831, only add
# things that are guaranteed safe improvements.
#
# Kept from original (proven):
#   ✓ WideResNet-28-10
#   ✓ RandAugment(n=2, m=9)  ← NOT replaced with AugMix
#   ✓ LR = 0.1               ← NOT scaled with batch size
#   ✓ CosineAnnealingWarmRestarts(T0=50)
#   ✓ Mixup + CutMix + RandomErasing
#   ✓ WeightedRandomSampler
#   ✓ TTA × 10
#
# Safe additions only:
#   + 200 epochs (was 100) with SWA over last 40
#   + DropPath=0.1 inside WideResNet blocks
#   + DataParallel across both T4s
#   + Proper 5% val split (was val=train which was a bug)
#   + All FutureWarnings fixed (torch.amp)
#   + Mixed precision (AMP) for speed
#
# Expected: 0.88–0.91  (from 0.831 baseline)
# Time:     ~7.5 hrs on T4 x2  (fits 12hr session)
# =============================================================================

import os, random, time, math
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import GradScaler, autocast          # ✓ no FutureWarning
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.transforms import RandAugment, RandomErasing  # ← proven, kept

# ── 0. GPU Setup ──────────────────────────────────────────────────────────────
print(f"GPUs available : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_GPUS  = torch.cuda.device_count()
USE_MULTI = NUM_GPUS > 1
print(f"Mode: {'DataParallel — both T4s active' if USE_MULTI else 'Single GPU'}\n")

# ── 1. Config ─────────────────────────────────────────────────────────────────
class CFG:
    seed         = 42
    num_classes  = 10
    batch_size   = 256 if USE_MULTI else 128   # double batch for dual GPU
    num_workers  = 4
    epochs       = 200
    swa_start    = 160
    swa_lr       = 0.005                       # conservative SWA LR

    # ✓ LR stays at 0.1 — linear scaling rule does NOT help at this batch size
    lr           = 0.1
    momentum     = 0.9
    weight_decay = 5e-4
    nesterov     = True

    # Cosine warm restarts — same as original working code
    T0           = 50
    T_mult       = 1
    eta_min      = 1e-6

    # 5% val split — small enough to keep training data, real enough to signal
    val_frac     = 0.05

    # RandAugment — proven to work, NOT replaced
    randaug_n    = 2
    randaug_m    = 9

    # Mixup / CutMix
    mixup_alpha  = 0.4
    cutmix_alpha = 1.0
    mix_prob     = 0.5
    label_smooth = 0.1

    # TTA
    tta_n        = 10

    # Paths
    base_dir     = Path('/kaggle/input/competitions/shift-guard-10-robust-image-classification-challenge')
    train_dir    = base_dir / 'train_images'
    test_dir     = base_dir / 'test_images'
    train_csv    = base_dir / 'train_labels.csv'
    sample_sub   = base_dir / 'sample_submission.csv'
    out_dir      = Path('/kaggle/working')

est = CFG.epochs * (130 if USE_MULTI else 240) / 60
print(f"Batch size : {CFG.batch_size}")
print(f"LR         : {CFG.lr}  (fixed — not scaled)")
print(f"Epochs     : {CFG.epochs}  |  SWA from epoch {CFG.swa_start}")
print(f"Val split  : {int(CFG.val_frac*100)}%  (~{int(29400*CFG.val_frac):,} held out)")
print(f"Est. time  : {est:.0f} min  ({est/60:.1f} hrs)\n")

# ── 2. Reproducibility ────────────────────────────────────────────────────────
def seed_everything(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    os.environ['PYTHONHASHSEED'] = str(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

seed_everything(CFG.seed)

CLASSES = ['airplane','automobile','bird','cat','deer',
           'dog','frog','horse','ship','truck']
CLS2IDX = {c: i for i, c in enumerate(CLASSES)}
IDX2CLS = {i: c for c, i in CLS2IDX.items()}
MEAN    = [0.4914, 0.4822, 0.4465]
STD     = [0.2470, 0.2435, 0.2616]

# ── 3. Transforms (RandAugment kept — proven) ─────────────────────────────────
def build_train_transforms():
    return transforms.Compose([
        transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
        transforms.RandomHorizontalFlip(),
        RandAugment(num_ops=CFG.randaug_n, magnitude=CFG.randaug_m),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
        RandomErasing(p=0.5, scale=(0.02, 0.2), ratio=(0.3, 3.3), value='random'),
    ])

def build_val_transforms():
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])

def build_tta_transforms():
    tfms    = []
    offsets = [(0,0),(0,8),(8,0),(8,8),(4,4)]
    for flip in [False, True]:
        for r, c in offsets:
            ops = []
            if flip:
                ops.append(transforms.RandomHorizontalFlip(p=1.0))
            ops.append(transforms.Pad(4, padding_mode='reflect'))
            ops.append(transforms.Lambda(
                lambda img, r=r, c=c: transforms.functional.crop(img, r, c, 32, 32)))
            ops += [transforms.ToTensor(), transforms.Normalize(MEAN, STD)]
            tfms.append(transforms.Compose(ops))
    return tfms[:CFG.tta_n]

TTA_TRANSFORMS = build_tta_transforms()

# ── 4. Dataset ────────────────────────────────────────────────────────────────
class ImageDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, labelled=True):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.labelled  = labelled
        # pre-resolve paths once at init
        self.paths = []
        for _, row in self.df.iterrows():
            img_id = str(row['id']).zfill(6)
            p      = img_dir / f'{img_id}.png'
            if not p.exists():
                p = img_dir / f"{row['id']}.png"
            self.paths.append(p)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        if self.labelled:
            return img, int(self.df.iloc[idx]['label_idx'])
        return img, str(self.df.iloc[idx]['id'])

# ── 5. WideResNet-28-10 + DropPath (new, safe addition) ──────────────────────
class DropPath(nn.Module):
    """Stochastic depth — drops entire residual branch randomly in training."""
    def __init__(self, p=0.1):
        super().__init__(); self.p = p
    def forward(self, x):
        if not self.training or self.p == 0:
            return x
        keep = 1 - self.p
        mask = torch.rand(x.shape[0], 1, 1, 1, device=x.device) < keep
        return x * mask.float() / keep

class _WideBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride, drop=0.3, drop_path=0.1):
        super().__init__()
        self.bn1  = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn2  = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.drop = drop
        self.dp   = DropPath(drop_path)
        self.skip = None if (in_ch == out_ch and stride == 1) else \
                    nn.Conv2d(in_ch, out_ch, 1, stride, 0, bias=False)

    def forward(self, x):
        h   = F.relu(self.bn1(x), inplace=True)
        out = self.conv1(h)
        out = F.relu(self.bn2(out), inplace=True)
        if self.drop > 0 and self.training:
            out = F.dropout(out, p=self.drop)
        out = self.dp(self.conv2(out))
        return out + (self.skip(h) if self.skip else x)

class _WideGroup(nn.Module):
    def __init__(self, n, in_ch, out_ch, stride, drop=0.3, drop_path=0.1):
        super().__init__()
        layers = [_WideBlock(in_ch, out_ch, stride, drop, drop_path)]
        for _ in range(n - 1):
            layers.append(_WideBlock(out_ch, out_ch, 1, drop, drop_path))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class WideResNet(nn.Module):
    def __init__(self, depth=28, width=10, num_classes=10, drop=0.3, drop_path=0.1):
        super().__init__()
        assert (depth - 4) % 6 == 0
        n   = (depth - 4) // 6
        chs = [16, 16*width, 32*width, 64*width]
        self.conv0  = nn.Conv2d(3, chs[0], 3, 1, 1, bias=False)
        self.group1 = _WideGroup(n, chs[0], chs[1], 1, drop, drop_path)
        self.group2 = _WideGroup(n, chs[1], chs[2], 2, drop, drop_path)
        self.group3 = _WideGroup(n, chs[2], chs[3], 2, drop, drop_path)
        self.bn     = nn.BatchNorm2d(chs[3])
        self.fc     = nn.Linear(chs[3], num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.conv0(x)
        x = self.group1(x); x = self.group2(x); x = self.group3(x)
        x = F.relu(self.bn(x), inplace=True)
        return self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))

def build_model():
    m = WideResNet(28, 10, CFG.num_classes, drop=0.3, drop_path=0.1).to(DEVICE)
    if USE_MULTI:
        m = nn.DataParallel(m)
        print(f"  DataParallel across {NUM_GPUS} GPUs")
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"  Params: {n/1e6:.2f}M")
    return m

# ── 6. Loss ───────────────────────────────────────────────────────────────────
class SoftCE(nn.Module):
    def __init__(self, eps=0.1):
        super().__init__(); self.eps = eps
    def forward(self, logits, targets):
        C = logits.size(1)
        if targets.dim() == 1:
            targets = F.one_hot(targets, C).float()
        targets = (1 - self.eps) * targets + self.eps / C
        return -(targets * F.log_softmax(logits, 1)).sum(1).mean()

criterion = SoftCE(CFG.label_smooth)

# ── 7. Mixup / CutMix ────────────────────────────────────────────────────────
def rand_bbox(size, lam):
    H, W = size[-2], size[-1]
    r    = math.sqrt(1 - lam)
    h, w = int(H*r), int(W*r)
    cy, cx = random.randint(0, H), random.randint(0, W)
    return max(0,cx-w//2), max(0,cy-h//2), min(W,cx+w//2), min(H,cy+h//2)

def apply_mix(imgs, lbls):
    perm = torch.randperm(imgs.size(0), device=imgs.device)
    oh   = F.one_hot(lbls, CFG.num_classes).float()
    if random.random() < 0.5:
        lam  = np.random.beta(CFG.mixup_alpha, CFG.mixup_alpha)
        imgs = lam * imgs + (1-lam) * imgs[perm]
    else:
        lam = np.random.beta(CFG.cutmix_alpha, CFG.cutmix_alpha)
        x1,y1,x2,y2 = rand_bbox(imgs.shape, lam)
        imgs = imgs.clone()
        imgs[:,:,y1:y2,x1:x2] = imgs[perm,:,y1:y2,x1:x2]
        lam = 1 - (x2-x1)*(y2-y1)/(imgs.shape[-1]*imgs.shape[-2])
    return imgs, lam*oh + (1-lam)*oh[perm]

# ── 8. Weighted Sampler ───────────────────────────────────────────────────────
def make_sampler(label_list):
    counts  = np.bincount(label_list, minlength=CFG.num_classes).astype(float)
    weights = 1.0 / (counts + 1e-6)
    sw      = torch.tensor([weights[l] for l in label_list], dtype=torch.float)
    return WeightedRandomSampler(sw, len(sw), replacement=True)

# ── 9. Train one epoch ────────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    running = 0.0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        if random.random() < CFG.mix_prob:
            imgs, soft = apply_mix(imgs, lbls)
        else:
            soft = lbls
        optimizer.zero_grad()
        with autocast('cuda'):
            loss = criterion(model(imgs), soft)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        params = model.module.parameters() if USE_MULTI else model.parameters()
        nn.utils.clip_grad_norm_(params, 5.0)
        scaler.step(optimizer); scaler.update()
        running += loss.item()
    return running / len(loader)

# ── 10. Validate ──────────────────────────────────────────────────────────────
@torch.no_grad()
def validate(model, loader):
    model.eval()
    preds, trues = [], []
    for imgs, lbls in loader:
        imgs = imgs.to(DEVICE)
        with autocast('cuda'):
            p = model(imgs).argmax(1).cpu().numpy()
        preds.extend(p); trues.extend(lbls.numpy())
    return (f1_score(trues, preds, average='macro'),
            np.mean(np.array(preds) == np.array(trues)))

# ── 11. TTA Inference ─────────────────────────────────────────────────────────
@torch.no_grad()
def predict_tta(model, df, img_dir):
    model.eval()
    all_probs = []
    for i, tf in enumerate(TTA_TRANSFORMS, 1):
        ds = ImageDataset(df, img_dir, transform=tf, labelled=False)
        ld = DataLoader(ds, CFG.batch_size*2, num_workers=CFG.num_workers,
                        shuffle=False, pin_memory=True)
        probs = []
        for imgs, _ in ld:
            with autocast('cuda'):
                logits = model(imgs.to(DEVICE))
            probs.append(F.softmax(logits, 1).cpu().float().numpy())
        all_probs.append(np.concatenate(probs, 0))
        print(f"    TTA {i}/{CFG.tta_n}", end='\r')
    print()
    return np.mean(all_probs, 0)

# ── 12. Load & Split Data ─────────────────────────────────────────────────────
print('Loading data …')
train_df = pd.read_csv(CFG.train_csv)
train_df.columns = [c.strip().lower() for c in train_df.columns]
if 'label' in train_df.columns:
    train_df['label_idx'] = train_df['label'].map(CLS2IDX)
elif 'class' in train_df.columns:
    train_df['label_idx'] = train_df['class'].map(CLS2IDX)

test_df = pd.read_csv(CFG.sample_sub)
test_df.columns = [c.strip().lower() for c in test_df.columns]

# 5% real validation split — small but honest
tr_df, va_df = train_test_split(
    train_df, test_size=CFG.val_frac,
    stratify=train_df['label_idx'], random_state=CFG.seed
)
tr_df = tr_df.reset_index(drop=True)
va_df = va_df.reset_index(drop=True)
print(f"Train : {len(tr_df):,}  |  Val : {len(va_df):,}  |  Test : {len(test_df):,}")

# ── 13. DataLoaders ───────────────────────────────────────────────────────────
tr_ds   = ImageDataset(tr_df, CFG.train_dir, build_train_transforms(), labelled=True)
va_ds   = ImageDataset(va_df, CFG.train_dir, build_val_transforms(),   labelled=True)

sampler = make_sampler(tr_df['label_idx'].tolist())
tr_ld   = DataLoader(tr_ds, CFG.batch_size, sampler=sampler,
                     num_workers=CFG.num_workers, pin_memory=True, drop_last=True)
va_ld   = DataLoader(va_ds, CFG.batch_size*2, shuffle=False,
                     num_workers=CFG.num_workers, pin_memory=True)

# ── 14. Model / Optimizer / Scheduler ────────────────────────────────────────
print('\nBuilding model …')
model     = build_model()
raw_model = model.module if USE_MULTI else model
raw_params = raw_model.parameters()

optimizer = optim.SGD(raw_params, lr=CFG.lr,
                      momentum=CFG.momentum, weight_decay=CFG.weight_decay,
                      nesterov=CFG.nesterov)

# Same scheduler as original working code — proven
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=CFG.T0, T_mult=CFG.T_mult, eta_min=CFG.eta_min
)

scaler    = GradScaler('cuda')

swa_model = AveragedModel(raw_model)
swa_sched = SWALR(optimizer, swa_lr=CFG.swa_lr, anneal_epochs=10)

# ── 15. Training Loop ─────────────────────────────────────────────────────────
best_f1   = 0.0
best_ckpt = CFG.out_dir / 'best_model.pth'
log_rows  = []
t_total   = time.time()

print(f"\n{'Epoch':>6} {'Loss':>8} {'Val F1':>8} {'Val Acc':>9} {'LR':>10} {'Time':>7}")
print('─' * 58)

for epoch in range(1, CFG.epochs + 1):
    t0 = time.time()

    tr_loss = train_one_epoch(model, tr_ld, optimizer, scaler)

    # scheduler: warm restarts until SWA kicks in
    if epoch < CFG.swa_start:
        scheduler.step(epoch)
    else:
        swa_model.update_parameters(raw_model)
        swa_sched.step()

    val_f1, val_acc = validate(model, va_ld)
    lr_now = optimizer.param_groups[0]['lr']

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(raw_model.state_dict(), best_ckpt)

    log_rows.append({'epoch': epoch, 'loss': tr_loss,
                     'val_f1': val_f1, 'val_acc': val_acc, 'lr': lr_now})

    if epoch % 10 == 0 or epoch <= 5:
        star = ' ★' if val_f1 >= best_f1 else ''
        dt   = time.time() - t0
        print(f'{epoch:>6} {tr_loss:>8.4f} {val_f1:>8.4f} '
              f'{val_acc*100:>8.2f}% {lr_now:>10.6f} {dt:>5.0f}s{star}')

elapsed = (time.time() - t_total) / 3600
print(f'\nTraining done in {elapsed:.2f} hrs  |  Best val F1 : {best_f1:.4f}')

# ── 16. SWA BN Update + Model Selection ──────────────────────────────────────
print('\nUpdating SWA batch-norm statistics …')
update_bn(tr_ld, swa_model, device=DEVICE)
swa_f1, swa_acc = validate(swa_model, va_ld)
print(f'SWA model  → Val F1 : {swa_f1:.4f}  |  Acc : {swa_acc*100:.2f}%')
print(f'Best epoch → Val F1 : {best_f1:.4f}')

if swa_f1 >= best_f1:
    infer_model = swa_model
    print('Using SWA model for inference.')
else:
    raw_model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
    infer_model = raw_model
    print(f'Using best checkpoint (F1={best_f1:.4f}) for inference.')

pd.DataFrame(log_rows).to_csv(CFG.out_dir / 'training_log.csv', index=False)

# ── 17. TTA Inference + Submission ───────────────────────────────────────────
print(f'\nRunning TTA inference ({CFG.tta_n} passes) …')
probs       = predict_tta(infer_model, test_df, CFG.test_dir)
pred_labels = [IDX2CLS[i] for i in probs.argmax(1)]

sub = pd.DataFrame({'id': test_df['id'], 'label': pred_labels})
sub.to_csv(CFG.out_dir / 'submission.csv', index=False)
print(f'\n✅  submission.csv saved  ({len(sub):,} rows)')
print(sub.head(10).to_string(index=False))
print('\nPredicted class distribution:')
print(sub['label'].value_counts().to_string())

GPUs available : 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4
Mode: DataParallel — both T4s active

Batch size : 256
LR         : 0.1  (fixed — not scaled)
Epochs     : 200  |  SWA from epoch 160
Val split  : 5%  (~1,470 held out)
Est. time  : 433 min  (7.2 hrs)

Loading data …
Train : 27,930  |  Val : 1,470  |  Test : 7,600

Building model …
  DataParallel across 2 GPUs
  Params: 36.48M

 Epoch     Loss   Val F1   Val Acc         LR    Time
──────────────────────────────────────────────────────────
     1   2.2022   0.1825    20.14%   0.099901    50s ★
     2   2.0417   0.2295    32.24%   0.099606    45s ★
     3   1.9649   0.2913    34.15%   0.099114    48s ★
     4   1.8594   0.3774    46.19%   0.098429    52s ★
     5   1.8102   0.3925    47.82%   0.097553    50s ★
    10   1.6129   0.4967    56.60%   0.090451    51s ★
    20   1.3886   0.6343    71.36%   0.065451    50s
    30   1.2430   0.7154    77.62%   0.034550    50s ★
    40   1.0724   0.7569    84.90%   0.009550    50s
    50   1.

## ShiftGuard10 — v2: AugMix + JSD Consistency Loss

_AugMix + JSD consistency loss | 20-pass TTA | **Best Val F1: 0.9231**_

---

In [1]:
# =============================================================================
# ShiftGuard10 — best_model_v2.py
# Strategy: Close the val→public gap with AugMix + JSD consistency loss
#
# Base (proven F1=0.9106 val / 0.8346 public):
#   my-notebook-3.ipynb → WRN-28-10, RandAugment, CosineWarmRestarts, SWA
#
# Key addition for 0.9+ PUBLIC score:
#   ✦ AugMix with JSD consistency loss (proven +3-5% on distribution-shift benchmarks)
#     For each training sample, model sees (clean, augmix). Loss penalises
#     inconsistent predictions between the two views. Forces learning
#     features invariant to corruptions, not just pixel patterns.
#
# Other improvements:
#   + 5-epoch linear LR warmup (stabilises JSD training)
#   + 20-pass TTA including blur/noise passes (not just crop/flip)
#   + Batch size 128 (fits clean+aug in same GPU memory slot as before)
#   + 200 epochs (JSD converges faster; fits 12h limit)
# =============================================================================

import os, random, time, math, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageEnhance, ImageFilter
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import GradScaler, autocast
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.transforms import RandAugment, RandomErasing
warnings.filterwarnings('ignore')

# ── 0. GPU Setup ──────────────────────────────────────────────────────────────
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_GPUS  = torch.cuda.device_count()
USE_MULTI = NUM_GPUS > 1
print(f"GPUs: {NUM_GPUS} | Mode: {'DataParallel' if USE_MULTI else 'Single GPU'}")
for i in range(NUM_GPUS):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
print()

# ── 1. Config ─────────────────────────────────────────────────────────────────
class CFG:
    seed         = 42
    num_classes  = 10
    # ↓ 128 instead of 256: each step combines clean+aug → same GPU memory as before
    batch_size   = 128
    num_workers  = 4
    epochs       = 200
    warmup_eps   = 5            # linear LR warmup epochs
    swa_start    = 160
    swa_lr       = 0.005

    lr           = 0.1          # NOT scaled — proven to work
    momentum     = 0.9
    weight_decay = 5e-4
    nesterov     = True
    grad_clip    = 5.0

    T0           = 50           # CosineAnnealingWarmRestarts period
    T_mult       = 1
    eta_min      = 1e-6

    val_frac     = 0.05

    randaug_n    = 2
    randaug_m    = 12           # stronger than proven 9, but controlled by JSD

    mixup_alpha  = 0.4
    cutmix_alpha = 1.0
    mix_prob     = 0.5
    label_smooth = 0.1          # back to 0.1 — JSD handles robustness now

    # JSD consistency loss weight (AugMix paper uses 12)
    lambda_jsd   = 12.0

    tta_n        = 20

    if os.path.exists('/kaggle/input'):
        base_dir = Path('/kaggle/input/competitions/shift-guard-10-robust-image-classification-challenge')
        out_dir  = Path('/kaggle/working')
    else:
        base_dir = Path(r'c:\Users\karti\Downloads\shiftgaurd\shift-guard-10-robust-image-classification-challenge')
        out_dir  = Path(r'c:\Users\karti\Downloads\shiftgaurd')

    train_dir  = base_dir / 'train_images'
    test_dir   = base_dir / 'test_images'
    train_csv  = base_dir / 'train_labels.csv'
    sample_sub = base_dir / 'sample_submission.csv'

est = CFG.epochs * (150 if USE_MULTI else 270) / 60
print(f"Batch size  : {CFG.batch_size}  (clean+aug combined → {CFG.batch_size*2} per step)")
print(f"LR          : {CFG.lr}  (fixed) | Warmup: {CFG.warmup_eps} epochs")
print(f"Epochs      : {CFG.epochs}  | SWA from {CFG.swa_start}")
print(f"JSD λ       : {CFG.lambda_jsd}  (AugMix consistency loss)")
print(f"TTA passes  : {CFG.tta_n}")
print(f"Est. time   : {est:.0f} min  ({est/60:.1f} hrs)\n")

# ── 2. Reproducibility ────────────────────────────────────────────────────────
def seed_everything(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    os.environ['PYTHONHASHSEED'] = str(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

seed_everything(CFG.seed)

CLASSES = ['airplane','automobile','bird','cat','deer',
           'dog','frog','horse','ship','truck']
CLS2IDX = {c: i for i, c in enumerate(CLASSES)}
IDX2CLS = {i: c for c, i in CLS2IDX.items()}
MEAN    = [0.4914, 0.4822, 0.4465]
STD     = [0.2470, 0.2435, 0.2616]

# ── 3. AugMix ─────────────────────────────────────────────────────────────────
def _int(level, maxval): return int(level * maxval / 10)
def _flt(level, maxval): return float(level) * maxval / 10.

AUG_OPS = [
    lambda img, l: ImageOps.autocontrast(img),
    lambda img, l: ImageOps.equalize(img),
    lambda img, l: ImageOps.posterize(img, max(1, 4 - _int(l, 4))),
    lambda img, l: ImageOps.solarize(img, 256 - _int(l, 256)),
    lambda img, l: img.rotate(_int(l, 30) * random.choice([-1, 1]), fillcolor=(128,128,128)),
    lambda img, l: img.transform(img.size, Image.AFFINE, (1, _flt(l,.3)*random.choice([-1,1]), 0, 0, 1, 0), fillcolor=(128,128,128)),
    lambda img, l: img.transform(img.size, Image.AFFINE, (1, 0, 0, _flt(l,.3)*random.choice([-1,1]), 1, 0), fillcolor=(128,128,128)),
    lambda img, l: ImageEnhance.Color(img).enhance(_flt(l, 1.8) + 0.1),
    lambda img, l: ImageEnhance.Contrast(img).enhance(_flt(l, 1.8) + 0.1),
    lambda img, l: ImageEnhance.Brightness(img).enhance(_flt(l, 1.8) + 0.1),
    lambda img, l: ImageEnhance.Sharpness(img).enhance(_flt(l, 1.8) + 0.1),
]

def augmix_pil(image, severity=3, width=3, alpha=1.):
    ws  = np.float32(np.random.dirichlet([alpha] * width))
    m   = np.float32(np.random.beta(alpha, alpha))
    mix = np.zeros_like(np.array(image), dtype=np.float32)
    for i in range(width):
        img_aug = image.copy()
        for _ in range(np.random.randint(1, 4)):
            img_aug = random.choice(AUG_OPS)(img_aug, np.random.randint(1, severity + 1))
        mix += ws[i] * np.array(img_aug, dtype=np.float32)
    return Image.fromarray(np.uint8(np.clip((1-m)*np.array(image, np.float32) + m*mix, 0, 255)))

# ── 4. Transforms ─────────────────────────────────────────────────────────────
# Clean transform: standard augmentation WITHOUT AugMix
clean_tfm = transforms.Compose([
    transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Augmix transform: AugMix applied in __getitem__ + RandAugment on top
augmix_base_tfm = transforms.Compose([
    transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
    transforms.RandomHorizontalFlip(),
    RandAugment(num_ops=CFG.randaug_n, magnitude=CFG.randaug_m),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    RandomErasing(p=0.4, scale=(0.02, 0.2), ratio=(0.3, 3.3), value='random'),
])

val_tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

def build_tta_transforms():
    """
    20 diverse TTA passes:
      - 10 crop/flip variants (proven, same as notebook)
      - 5 with light color jitter
      - 5 with slight blur (simulates sensor/focus shift)
    """
    tfms = []
    to_norm = [transforms.ToTensor(), transforms.Normalize(MEAN, STD)]
    offsets = [(0,0),(0,8),(8,0),(8,8),(4,4),(2,6),(6,2),(0,4),(4,0),(4,8)]

    # Group 1: crop/flip (10 passes)
    for flip in [False, True]:
        for r, c in offsets[:5]:
            ops = []
            if flip: ops.append(transforms.RandomHorizontalFlip(p=1.0))
            ops.append(transforms.Pad(4, padding_mode='reflect'))
            ops.append(transforms.Lambda(
                lambda img, r=r, c=c: transforms.functional.crop(img, r, c, 32, 32)))
            ops += to_norm
            tfms.append(transforms.Compose(ops))

    # Group 2: crop/flip + light color jitter (5 passes)
    jitter = transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15)
    for flip in [False, True]:
        for r, c in [(0,0),(4,4),(8,8)]:
            ops = []
            if flip: ops.append(transforms.RandomHorizontalFlip(p=1.0))
            ops += [jitter, transforms.Pad(4, padding_mode='reflect')]
            ops.append(transforms.Lambda(
                lambda img, r=r, c=c: transforms.functional.crop(img, r, c, 32, 32)))
            ops += to_norm
            tfms.append(transforms.Compose(ops))

    # Group 3: crop/flip + slight Gaussian blur (5 passes)
    blur = transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5))
    for flip in [False, True]:
        for r, c in [(0,0),(4,4),(8,8)]:
            ops = []
            if flip: ops.append(transforms.RandomHorizontalFlip(p=1.0))
            ops += [blur, transforms.Pad(4, padding_mode='reflect')]
            ops.append(transforms.Lambda(
                lambda img, r=r, c=c: transforms.functional.crop(img, r, c, 32, 32)))
            ops += to_norm
            tfms.append(transforms.Compose(ops))

    return tfms[:CFG.tta_n]

TTA_TRANSFORMS = build_tta_transforms()

# ── 5. Dataset ─────────────────────────────────────────────────────────────────
class JSDDataset(Dataset):
    """
    Returns (x_clean, x_aug, label) for AugMix JSD training.
    x_clean: standard crop+flip only
    x_aug  : AugMix + RandAugment (diverse corruption-like augmentation)
    """
    def __init__(self, df, img_dir, labelled=True):
        self.df       = df.reset_index(drop=True)
        self.labelled = labelled
        self.paths    = []
        for _, row in self.df.iterrows():
            img_id = str(row['id']).zfill(6)
            p      = img_dir / f'{img_id}.png'
            if not p.exists():
                p = img_dir / f"{row['id']}.png"
            self.paths.append(p)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.labelled:
            x_clean = clean_tfm(img)
            x_aug   = augmix_base_tfm(augmix_pil(img))  # AugMix first, then RandAugment
            label   = int(self.df.iloc[idx]['label_idx'])
            return x_clean, x_aug, label
        else:
            return val_tfm(img), str(self.df.iloc[idx]['id'])

class ValDataset(Dataset):
    def __init__(self, df, img_dir):
        self.df    = df.reset_index(drop=True)
        self.paths = [img_dir / f"{str(r['id']).zfill(6)}.png" for _, r in df.iterrows()]
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        return val_tfm(img), int(self.df.iloc[idx]['label_idx'])

class TestDataset(Dataset):
    def __init__(self, df, img_dir, tfm):
        self.df    = df.reset_index(drop=True)
        self.tfm   = tfm
        self.paths = [img_dir / f"{str(r['id']).zfill(6)}.png" for _, r in df.iterrows()]
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.tfm: img = self.tfm(img)
        return img, str(self.df.iloc[idx]['id'])

# ── 6. Model: WideResNet-28-10 + DropPath ────────────────────────────────────
class DropPath(nn.Module):
    def __init__(self, p=0.1):
        super().__init__(); self.p = p
    def forward(self, x):
        if not self.training or self.p == 0: return x
        keep = 1 - self.p
        return x * (torch.rand(x.shape[0],1,1,1, device=x.device) < keep).float() / keep

class _WideBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride, drop=0.3, drop_path=0.1):
        super().__init__()
        self.bn1   = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.drop  = drop
        self.dp    = DropPath(drop_path)
        self.skip  = None if (in_ch == out_ch and stride == 1) else \
                     nn.Conv2d(in_ch, out_ch, 1, stride, 0, bias=False)

    def forward(self, x):
        h   = F.relu(self.bn1(x))             # NO inplace — AMP safe
        out = self.conv1(h)
        out = F.relu(self.bn2(out))            # NO inplace — AMP safe
        if self.drop > 0 and self.training:
            out = F.dropout(out, p=self.drop, training=True)
        out = self.dp(self.conv2(out))
        return out + (self.skip(h) if self.skip else x)

class _WideGroup(nn.Module):
    def __init__(self, n, in_ch, out_ch, stride, drop=0.3, drop_path=0.1):
        super().__init__()
        layers = [_WideBlock(in_ch, out_ch, stride, drop, drop_path)]
        for _ in range(n - 1):
            layers.append(_WideBlock(out_ch, out_ch, 1, drop, drop_path))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class WideResNet(nn.Module):
    def __init__(self, depth=28, width=10, num_classes=10, drop=0.3, drop_path=0.1):
        super().__init__()
        assert (depth - 4) % 6 == 0
        n   = (depth - 4) // 6
        chs = [16, 16*width, 32*width, 64*width]
        self.conv0  = nn.Conv2d(3, chs[0], 3, 1, 1, bias=False)
        self.group1 = _WideGroup(n, chs[0], chs[1], 1, drop, drop_path)
        self.group2 = _WideGroup(n, chs[1], chs[2], 2, drop, drop_path)
        self.group3 = _WideGroup(n, chs[2], chs[3], 2, drop, drop_path)
        self.bn     = nn.BatchNorm2d(chs[3])
        self.fc     = nn.Linear(chs[3], num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.conv0(x)
        x = self.group1(x); x = self.group2(x); x = self.group3(x)
        x = F.relu(self.bn(x))                # NO inplace — AMP safe
        return self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))

def build_model():
    m = WideResNet(28, 10, CFG.num_classes, drop=0.3, drop_path=0.1).to(DEVICE)
    if USE_MULTI:
        m = nn.DataParallel(m)
        print(f"  DataParallel across {NUM_GPUS} GPUs")
    print(f"  Params: {sum(p.numel() for p in m.parameters() if p.requires_grad)/1e6:.2f}M")
    return m

# ── 7. Loss ───────────────────────────────────────────────────────────────────
class SoftCE(nn.Module):
    def __init__(self, eps=0.1):
        super().__init__(); self.eps = eps
    def forward(self, logits, targets):
        C = logits.size(1)
        if targets.dim() == 1:
            targets = F.one_hot(targets, C).float()
        targets = (1 - self.eps) * targets + self.eps / C
        return -(targets * F.log_softmax(logits, 1)).sum(1).mean()

def jsd_loss(logits_clean, logits_aug):
    """
    Jensen-Shannon Divergence consistency loss (from AugMix paper).
    Forces model to predict consistently on clean vs augmented images.
    This is the key to closing the distribution-shift gap.
    """
    p_clean = F.softmax(logits_clean.float(), dim=1)
    p_aug   = F.softmax(logits_aug.float(), dim=1)
    p_mix   = (p_clean + p_aug) / 2.0
    loss = (F.kl_div(p_mix.log(), p_clean, reduction='batchmean') +
            F.kl_div(p_mix.log(), p_aug,   reduction='batchmean')) / 2.0
    return loss

criterion = SoftCE(CFG.label_smooth)

# ── 8. Mixup / CutMix ─────────────────────────────────────────────────────────
def rand_bbox(size, lam):
    H, W = size[-2], size[-1]
    r    = math.sqrt(1 - lam)
    h, w = int(H*r), int(W*r)
    cy, cx = random.randint(0, H), random.randint(0, W)
    return max(0,cx-w//2), max(0,cy-h//2), min(W,cx+w//2), min(H,cy+h//2)

def apply_mix(imgs, lbls):
    perm = torch.randperm(imgs.size(0), device=imgs.device)
    oh   = F.one_hot(lbls, CFG.num_classes).float()
    if random.random() < 0.5:
        lam  = np.random.beta(CFG.mixup_alpha, CFG.mixup_alpha)
        imgs = lam * imgs + (1-lam) * imgs[perm]
    else:
        lam = np.random.beta(CFG.cutmix_alpha, CFG.cutmix_alpha)
        x1,y1,x2,y2 = rand_bbox(imgs.shape, lam)
        imgs = imgs.clone()
        imgs[:,:,y1:y2,x1:x2] = imgs[perm,:,y1:y2,x1:x2]
        lam = 1 - (x2-x1)*(y2-y1)/(imgs.shape[-1]*imgs.shape[-2])
    return imgs, lam*oh + (1-lam)*oh[perm]

# ── 9. Weighted Sampler ───────────────────────────────────────────────────────
def make_sampler(label_list):
    counts  = np.bincount(label_list, minlength=CFG.num_classes).astype(float)
    weights = 1.0 / (counts + 1e-6)
    sw      = torch.tensor([weights[l] for l in label_list], dtype=torch.float)
    return WeightedRandomSampler(sw, len(sw), replacement=True)

# ── 10. Train one epoch (with JSD consistency loss) ───────────────────────────
def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    running_ce = running_jsd = 0.0
    n_batches  = 0

    for x_clean, x_aug, lbls in loader:
        x_clean = x_clean.to(DEVICE, non_blocking=True)
        x_aug   = x_aug.to(DEVICE, non_blocking=True)
        lbls    = lbls.to(DEVICE, non_blocking=True)
        B       = x_clean.size(0)

        # Apply mixup/cutmix to clean images only
        if random.random() < CFG.mix_prob:
            x_clean_m, soft = apply_mix(x_clean, lbls)
        else:
            x_clean_m = x_clean
            soft      = lbls

        optimizer.zero_grad(set_to_none=True)

        with autocast('cuda'):
            # Single forward pass with both views concatenated → memory efficient
            all_logits   = model(torch.cat([x_clean_m, x_aug], dim=0))
            logits_clean = all_logits[:B]
            logits_aug   = all_logits[B:]

            loss_ce  = criterion(logits_clean, soft)
            loss_jsd = jsd_loss(logits_clean, logits_aug)
            loss     = loss_ce + CFG.lambda_jsd * loss_jsd

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        params = model.module.parameters() if USE_MULTI else model.parameters()
        nn.utils.clip_grad_norm_(params, CFG.grad_clip)
        scaler.step(optimizer); scaler.update()

        running_ce  += loss_ce.item()
        running_jsd += loss_jsd.item()
        n_batches   += 1

    return running_ce / n_batches, running_jsd / n_batches

# ── 11. Validate ──────────────────────────────────────────────────────────────
@torch.no_grad()
def validate(model, loader):
    model.eval()
    preds, trues = [], []
    for imgs, lbls in loader:
        imgs = imgs.to(DEVICE)
        with autocast('cuda'):
            p = model(imgs).argmax(1).cpu().numpy()
        preds.extend(p); trues.extend(lbls.numpy())
    return (f1_score(trues, preds, average='macro'),
            np.mean(np.array(preds) == np.array(trues)))

# ── 12. TTA Inference ─────────────────────────────────────────────────────────
@torch.no_grad()
def predict_tta(model, df, img_dir):
    model.eval()
    all_probs = []
    for i, tf in enumerate(TTA_TRANSFORMS, 1):
        ds = TestDataset(df, img_dir, tf)
        ld = DataLoader(ds, CFG.batch_size*2, num_workers=CFG.num_workers,
                        shuffle=False, pin_memory=True)
        probs = []
        for imgs, _ in ld:
            with autocast('cuda'):
                logits = model(imgs.to(DEVICE))
            probs.append(F.softmax(logits, 1).cpu().float().numpy())
        all_probs.append(np.concatenate(probs, 0))
        print(f"    TTA {i}/{CFG.tta_n}", end='\r')
    print()
    return np.mean(all_probs, 0)

# ── 13. Load & Split Data ─────────────────────────────────────────────────────
print('Loading data …')
train_df = pd.read_csv(CFG.train_csv)
train_df.columns = [c.strip().lower() for c in train_df.columns]
if 'label' in train_df.columns:
    train_df['label_idx'] = train_df['label'].map(CLS2IDX)
elif 'class' in train_df.columns:
    train_df['label_idx'] = train_df['class'].map(CLS2IDX)

test_df = pd.read_csv(CFG.sample_sub)
test_df.columns = [c.strip().lower() for c in test_df.columns]

tr_df, va_df = train_test_split(
    train_df, test_size=CFG.val_frac,
    stratify=train_df['label_idx'], random_state=CFG.seed
)
tr_df = tr_df.reset_index(drop=True)
va_df = va_df.reset_index(drop=True)
print(f"Train : {len(tr_df):,}  |  Val : {len(va_df):,}  |  Test : {len(test_df):,}\n")

# ── 14. DataLoaders ───────────────────────────────────────────────────────────
tr_ds   = JSDDataset(tr_df, CFG.train_dir, labelled=True)
va_ds   = ValDataset(va_df, CFG.train_dir)
sampler = make_sampler(tr_df['label_idx'].tolist())
tr_ld   = DataLoader(tr_ds, CFG.batch_size, sampler=sampler,
                     num_workers=CFG.num_workers, pin_memory=True, drop_last=True)
va_ld   = DataLoader(va_ds, CFG.batch_size*4, shuffle=False,
                     num_workers=CFG.num_workers, pin_memory=True)

# ── 15. Model / Optimizer / Scheduler ────────────────────────────────────────
print('Building model …')
model     = build_model()
raw_model = model.module if USE_MULTI else model

optimizer = optim.SGD(raw_model.parameters(), lr=CFG.lr,
                      momentum=CFG.momentum, weight_decay=CFG.weight_decay,
                      nesterov=CFG.nesterov)

# Linear warmup for warmup_eps epochs, then cosine warm restarts
def lr_lambda(epoch):
    if epoch < CFG.warmup_eps:
        return (epoch + 1) / CFG.warmup_eps   # linear 0 → 1
    return 1.0

warmup_sched  = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
cosine_sched  = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=CFG.T0, T_mult=CFG.T_mult, eta_min=CFG.eta_min
)

scaler    = GradScaler('cuda')
swa_model = AveragedModel(raw_model)
swa_sched = SWALR(optimizer, swa_lr=CFG.swa_lr, anneal_epochs=10)

# ── 16. Training Loop ─────────────────────────────────────────────────────────
best_f1   = 0.0
best_ckpt = CFG.out_dir / 'best_model.pth'
log_rows  = []
t_total   = time.time()

print(f"\n{'Ep':>4} {'CE':>7} {'JSD':>7} {'F1':>7} {'Acc':>7} {'LR':>9} {'Time':>6}")
print('─' * 58)

for epoch in range(1, CFG.epochs + 1):
    t0 = time.time()

    loss_ce, loss_jsd = train_one_epoch(model, tr_ld, optimizer, scaler)

    if epoch <= CFG.warmup_eps:
        warmup_sched.step()
    elif epoch < CFG.swa_start:
        cosine_sched.step(epoch - CFG.warmup_eps)
    else:
        swa_model.update_parameters(raw_model)
        swa_sched.step()

    val_f1, val_acc = validate(model, va_ld)
    lr_now = optimizer.param_groups[0]['lr']

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(raw_model.state_dict(), best_ckpt)

    log_rows.append({'epoch': epoch, 'loss_ce': loss_ce, 'loss_jsd': loss_jsd,
                     'val_f1': val_f1, 'val_acc': val_acc, 'lr': lr_now})

    if epoch % 10 == 0 or epoch <= 5:
        star = ' ★' if val_f1 >= best_f1 else ''
        dt   = time.time() - t0
        print(f'{epoch:>4} {loss_ce:>7.4f} {loss_jsd:>7.4f} {val_f1:>7.4f} '
              f'{val_acc*100:>6.2f}% {lr_now:>9.6f} {dt:>5.0f}s{star}')

elapsed = (time.time() - t_total) / 3600
print(f'\nTraining done in {elapsed:.2f} hrs  |  Best val F1 : {best_f1:.4f}')
print(f'(val F1 is expected ~0.02-0.03 lower than notebook-3 due to JSD regularisation)')

# ── 17. SWA BN Update ────────────────────────────────────────────────────────
print('\nUpdating SWA batch-norm statistics …')
# SWA update_bn expects (img,) or (img,label) — use a simple wrapper
class _SWALoader:
    def __init__(self, loader): self.loader = loader
    def __iter__(self):
        for x_clean, x_aug, lbl in self.loader:
            yield x_clean, lbl   # SWA only needs images for BN update

update_bn(_SWALoader(tr_ld), swa_model, device=DEVICE)
swa_f1, swa_acc = validate(swa_model, va_ld)
print(f'SWA model  → Val F1 : {swa_f1:.4f}  |  Acc : {swa_acc*100:.2f}%')
print(f'Best epoch → Val F1 : {best_f1:.4f}')

if swa_f1 >= best_f1:
    infer_model = swa_model
    print('Using SWA model for inference.')
else:
    raw_model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE, weights_only=True))
    infer_model = raw_model
    print(f'Using best checkpoint (F1={best_f1:.4f}) for inference.')

torch.save(raw_model.state_dict(), CFG.out_dir / 'final_model.pth')
pd.DataFrame(log_rows).to_csv(CFG.out_dir / 'training_log.csv', index=False)

# ── 18. TTA Inference + Submission ───────────────────────────────────────────
print(f'\nRunning TTA inference ({CFG.tta_n} passes) …')
probs       = predict_tta(infer_model, test_df, CFG.test_dir)
pred_labels = [IDX2CLS[i] for i in probs.argmax(1)]

sub = pd.DataFrame({'id': test_df['id'], 'label': pred_labels})
sub.to_csv(CFG.out_dir / 'submission.csv', index=False)
print(f'\n✅  submission.csv saved  ({len(sub):,} rows)')
print(sub.head(10).to_string(index=False))
print('\nPredicted class distribution:')
print(sub['label'].value_counts().to_string())
print(f'\nTotal time: {(time.time()-t_total)/3600:.2f} hrs')


GPUs: 2 | Mode: DataParallel
  GPU 0: Tesla T4
  GPU 1: Tesla T4

Batch size  : 128  (clean+aug combined → 256 per step)
LR          : 0.1  (fixed) | Warmup: 5 epochs
Epochs      : 200  | SWA from 160
JSD λ       : 12.0  (AugMix consistency loss)
TTA passes  : 20
Est. time   : 500 min  (8.3 hrs)

Loading data …
Train : 27,930  |  Val : 1,470  |  Test : 7,600

Building model …
  DataParallel across 2 GPUs
  Params: 36.48M

  Ep      CE     JSD      F1     Acc        LR   Time
──────────────────────────────────────────────────────────
   1  2.2664  0.0058  0.2488  34.08%  0.040000   102s ★
   2  2.0624  0.0084  0.2875  36.46%  0.060000   103s ★
   3  2.0140  0.0099  0.3097  35.58%  0.080000   104s ★
   4  1.9552  0.0115  0.4587  54.83%  0.100000   104s ★
   5  1.9307  0.0121  0.3874  48.78%  0.100000   103s
  10  1.7338  0.0166  0.5576  63.54%  0.097553   104s ★
  20  1.5647  0.0198  0.6452  73.13%  0.079389   102s
  30  1.4699  0.0229  0.7271  81.09%  0.050001   101s
  40  1.3872  0.024

## ShiftGuard10 — v3: Pseudo-Labeling + Corruption Augmentations

_Pseudo-labeling | Corruption augmentations | 30-pass TTA | **Best Val F1: 0.9262**_

---

In [1]:
# =============================================================================
# ShiftGuard10 — best_model_v3.py
# New vs v2 (0.847846 public):
#   ✦ Pseudo-labeling at epoch 100: model predicts on test set, confident
#     predictions (>0.9) added back to training → directly adapts to test dist
#   ✦ Explicit corruption augmentations: Gaussian noise, motion blur,
#     pixelation, JPEG artifacts → training data now looks like shifted test data
#   ✦ 30-pass TTA including corruption-aware passes
#   ✦ Soft pseudo-labels with temperature scaling (less overconfident)
#   ✦ Optional: load from saved v2 checkpoint to skip Phase 1
# =============================================================================

import os, io, random, time, math, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageEnhance, ImageFilter
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import GradScaler, autocast
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.transforms import RandAugment, RandomErasing
warnings.filterwarnings('ignore')

# ── 0. GPU Setup ──────────────────────────────────────────────────────────────
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_GPUS  = torch.cuda.device_count()
USE_MULTI = NUM_GPUS > 1
print(f"GPUs: {NUM_GPUS}")
for i in range(NUM_GPUS):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
print()

# ── 1. Config ─────────────────────────────────────────────────────────────────
class CFG:
    seed         = 42
    num_classes  = 10
    batch_size   = 128
    num_workers  = 4
    epochs       = 100          # fine-tuning from v2 checkpoint
    warmup_eps   = 2            # short warmup for fine-tuning
    swa_start    = 60
    swa_lr       = 0.001

    lr           = 0.01         # lower LR for fine-tuning (was 0.1 from scratch)
    momentum     = 0.9
    weight_decay = 5e-4
    nesterov     = True
    grad_clip    = 5.0

    T0           = 25           # shorter cosine cycle for fine-tuning
    T_mult       = 1
    eta_min      = 1e-6
    val_frac     = 0.05

    randaug_n    = 2
    randaug_m    = 12
    mixup_alpha  = 0.4
    cutmix_alpha = 1.0
    mix_prob     = 0.5
    label_smooth = 0.1
    lambda_jsd   = 12.0

    # Pseudo-labeling: generate BEFORE training starts (epoch 0)
    pseudo_epoch    = 0         # 0 = generate pseudo-labels immediately
    pseudo_conf     = 0.85      # lower threshold = more pseudo-labels accepted
    pseudo_tta      = 8         # more TTA passes for better pseudo-label quality
    pseudo_temp     = 2.0

    tta_n        = 30

    # v2 checkpoint loaded from Kaggle model input
    pretrained_ckpt = Path('/kaggle/input/models/kartikeyguptaiitk/trial-6-best/pytorch/default/1/best_model.pth')

    if os.path.exists('/kaggle/input'):
        base_dir = Path('/kaggle/input/competitions/shift-guard-10-robust-image-classification-challenge')
        out_dir  = Path('/kaggle/working')
    else:
        base_dir = Path(r'c:\Users\karti\Downloads\shiftgaurd\shift-guard-10-robust-image-classification-challenge')
        out_dir  = Path(r'c:\Users\karti\Downloads\shiftgaurd')

    train_dir  = base_dir / 'train_images'
    test_dir   = base_dir / 'test_images'
    train_csv  = base_dir / 'train_labels.csv'
    sample_sub = base_dir / 'sample_submission.csv'

CLASSES = ['airplane','automobile','bird','cat','deer',
           'dog','frog','horse','ship','truck']
CLS2IDX = {c: i for i, c in enumerate(CLASSES)}
IDX2CLS = {i: c for c, i in CLS2IDX.items()}
MEAN    = [0.4914, 0.4822, 0.4465]
STD     = [0.2470, 0.2435, 0.2616]

print(f"Batch    : {CFG.batch_size} (combined→{CFG.batch_size*2}/step)")
print(f"Epochs   : {CFG.epochs} | SWA from {CFG.swa_start}")
print(f"Pseudo   : epoch {CFG.pseudo_epoch}, conf>{CFG.pseudo_conf}, {CFG.pseudo_tta} TTA passes")
print(f"TTA      : {CFG.tta_n} passes")
est = CFG.epochs * 130 / 60 if USE_MULTI else CFG.epochs * 270 / 60
print(f"Est.time : {est:.0f} min ({est/60:.1f} hrs)\n")

# ── 2. Reproducibility ────────────────────────────────────────────────────────
def seed_everything(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    os.environ['PYTHONHASHSEED'] = str(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

seed_everything(CFG.seed)

# ── 3. Corruption augmentations (NEW — simulate test-set shifts) ──────────────
def gaussian_noise(img, std_range=(0.02, 0.12)):
    arr = np.array(img, dtype=np.float32) / 255.
    std = random.uniform(*std_range)
    arr = np.clip(arr + np.random.randn(*arr.shape) * std, 0, 1)
    return Image.fromarray((arr * 255).astype(np.uint8))

def motion_blur(img, max_size=5):
    size = random.choice([3, 5])
    return img.filter(ImageFilter.BoxBlur(size // 2))

def pixelate(img, factor=None):
    w, h = img.size
    f = factor or random.randint(2, 4)
    small = img.resize((max(1, w // f), max(1, h // f)), Image.BILINEAR)
    return small.resize((w, h), Image.NEAREST)

def jpeg_compress(img, quality_range=(20, 60)):
    quality = random.randint(*quality_range)
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=quality)
    buf.seek(0)
    return Image.open(buf).convert('RGB')

CORRUPTION_OPS = [gaussian_noise, motion_blur, pixelate, jpeg_compress]

def apply_corruption(img):
    """Apply 1 random corruption at 30% probability."""
    if random.random() < 0.3:
        return random.choice(CORRUPTION_OPS)(img)
    return img

# ── 4. AugMix ─────────────────────────────────────────────────────────────────
def _int(level, maxval): return int(level * maxval / 10)
def _flt(level, maxval): return float(level) * maxval / 10.

AUG_OPS = [
    lambda img, l: ImageOps.autocontrast(img),
    lambda img, l: ImageOps.equalize(img),
    lambda img, l: ImageOps.posterize(img, max(1, 4 - _int(l, 4))),
    lambda img, l: ImageOps.solarize(img, 256 - _int(l, 256)),
    lambda img, l: img.rotate(_int(l, 30) * random.choice([-1, 1]), fillcolor=(128,128,128)),
    lambda img, l: img.transform(img.size, Image.AFFINE, (1, _flt(l,.3)*random.choice([-1,1]), 0, 0, 1, 0), fillcolor=(128,128,128)),
    lambda img, l: img.transform(img.size, Image.AFFINE, (1, 0, 0, _flt(l,.3)*random.choice([-1,1]), 1, 0), fillcolor=(128,128,128)),
    lambda img, l: ImageEnhance.Color(img).enhance(_flt(l, 1.8) + 0.1),
    lambda img, l: ImageEnhance.Contrast(img).enhance(_flt(l, 1.8) + 0.1),
    lambda img, l: ImageEnhance.Brightness(img).enhance(_flt(l, 1.8) + 0.1),
    lambda img, l: ImageEnhance.Sharpness(img).enhance(_flt(l, 1.8) + 0.1),
    # NEW: corruption ops also in AugMix chain
    lambda img, l: gaussian_noise(img),
    lambda img, l: motion_blur(img),
    lambda img, l: pixelate(img),
]

def augmix_pil(image, severity=3, width=3, alpha=1.):
    ws  = np.float32(np.random.dirichlet([alpha] * width))
    m   = np.float32(np.random.beta(alpha, alpha))
    mix = np.zeros_like(np.array(image), dtype=np.float32)
    for i in range(width):
        img_aug = image.copy()
        for _ in range(np.random.randint(1, 4)):
            img_aug = random.choice(AUG_OPS)(img_aug, np.random.randint(1, severity + 1))
        mix += ws[i] * np.array(img_aug, dtype=np.float32)
    return Image.fromarray(np.uint8(np.clip((1-m)*np.array(image, np.float32) + m*mix, 0, 255)))

# ── 5. Transforms ─────────────────────────────────────────────────────────────
clean_tfm = transforms.Compose([
    transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

augmix_base_tfm = transforms.Compose([
    transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
    transforms.RandomHorizontalFlip(),
    RandAugment(num_ops=CFG.randaug_n, magnitude=CFG.randaug_m),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    RandomErasing(p=0.4, scale=(0.02, 0.2), ratio=(0.3, 3.3), value='random'),
])

val_tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

def build_tta_transforms():
    """30 TTA passes: crop/flip + color-jitter + blur + corruption."""
    tfms    = []
    to_norm = [transforms.ToTensor(), transforms.Normalize(MEAN, STD)]
    offsets = [(0,0),(0,8),(8,0),(8,8),(4,4),(2,6),(6,2),(0,4),(4,0),(4,8)]

    # 10 crop/flip passes
    for flip in [False, True]:
        for r, c in offsets[:5]:
            ops = []
            if flip: ops.append(transforms.RandomHorizontalFlip(p=1.0))
            ops.append(transforms.Pad(4, padding_mode='reflect'))
            ops.append(transforms.Lambda(lambda img,r=r,c=c: transforms.functional.crop(img,r,c,32,32)))
            ops += to_norm
            tfms.append(transforms.Compose(ops))

    # 8 crop/flip + color jitter passes
    jitter = transforms.ColorJitter(0.15, 0.15, 0.15)
    for flip in [False, True]:
        for r, c in [(0,0),(4,4),(8,8),(2,6)]:
            ops = []
            if flip: ops.append(transforms.RandomHorizontalFlip(p=1.0))
            ops += [jitter, transforms.Pad(4, padding_mode='reflect')]
            ops.append(transforms.Lambda(lambda img,r=r,c=c: transforms.functional.crop(img,r,c,32,32)))
            ops += to_norm
            tfms.append(transforms.Compose(ops))

    # 6 blur passes (simulate sensor blur shift)
    blur = transforms.GaussianBlur(3, sigma=(0.1, 0.5))
    for flip in [False, True]:
        for r, c in [(0,0),(4,4),(8,0)]:
            ops = []
            if flip: ops.append(transforms.RandomHorizontalFlip(p=1.0))
            ops += [blur, transforms.Pad(4, padding_mode='reflect')]
            ops.append(transforms.Lambda(lambda img,r=r,c=c: transforms.functional.crop(img,r,c,32,32)))
            ops += to_norm
            tfms.append(transforms.Compose(ops))

    # 6 noise passes (simulate sensor noise shift)
    for flip in [False, True]:
        for r, c in [(0,0),(4,4),(8,8)]:
            ops = [transforms.Lambda(lambda img: gaussian_noise(img, (0.01, 0.05)))]
            if flip: ops.insert(0, transforms.RandomHorizontalFlip(p=1.0))
            ops.append(transforms.Pad(4, padding_mode='reflect'))
            ops.append(transforms.Lambda(lambda img,r=r,c=c: transforms.functional.crop(img,r,c,32,32)))
            ops += to_norm
            tfms.append(transforms.Compose(ops))

    return tfms[:CFG.tta_n]

TTA_TRANSFORMS = build_tta_transforms()

# ── 6. Datasets ────────────────────────────────────────────────────────────────
def _resolve_path(img_dir, id_):
    p = img_dir / f"{str(id_).zfill(6)}.png"
    if not p.exists():
        p = img_dir / f"{id_}.png"
    return p

class JSDDataset(Dataset):
    """Returns (x_clean, x_aug, label). Supports optional corruption."""
    def __init__(self, df, img_dir, use_corruption=True):
        self.df   = df.reset_index(drop=True)
        self.dir  = img_dir
        self.use_c = use_corruption
        self.paths = [_resolve_path(img_dir, r['id']) for _, r in df.iterrows()]
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.use_c:
            img = apply_corruption(img)   # 30% chance of corruption on original
        x_clean = clean_tfm(img)
        x_aug   = augmix_base_tfm(augmix_pil(img))
        return x_clean, x_aug, int(self.df.iloc[idx]['label_idx'])

class CombinedJSDDataset(Dataset):
    """Combines train + pseudo-labeled test. Returns same (x_clean, x_aug, label)."""
    def __init__(self, orig_df, orig_dir, pseudo_df, pseudo_dir):
        self.items = []
        for _, row in orig_df.iterrows():
            self.items.append((_resolve_path(orig_dir, row['id']), int(row['label_idx'])))
        for _, row in pseudo_df.iterrows():
            self.items.append((_resolve_path(pseudo_dir, row['id']), int(row['label_idx'])))

    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        path, label = self.items[idx]
        img = Image.open(path).convert('RGB')
        img = apply_corruption(img)
        x_clean = clean_tfm(img)
        x_aug   = augmix_base_tfm(augmix_pil(img))
        return x_clean, x_aug, label

class ValDataset(Dataset):
    def __init__(self, df, img_dir):
        self.df    = df.reset_index(drop=True)
        self.paths = [_resolve_path(img_dir, r['id']) for _, r in df.iterrows()]
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        return val_tfm(Image.open(self.paths[idx]).convert('RGB')), int(self.df.iloc[idx]['label_idx'])

class TestDataset(Dataset):
    def __init__(self, df, img_dir, tfm=None):
        self.df    = df.reset_index(drop=True)
        self.tfm   = tfm
        self.paths = [_resolve_path(img_dir, r['id']) for _, r in df.iterrows()]
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.tfm: img = self.tfm(img)
        return img, str(self.df.iloc[idx]['id'])

# ── 7. Model: WideResNet-28-10 + DropPath ─────────────────────────────────────
class DropPath(nn.Module):
    def __init__(self, p=0.1):
        super().__init__(); self.p = p
    def forward(self, x):
        if not self.training or self.p == 0: return x
        keep = 1 - self.p
        return x * (torch.rand(x.shape[0],1,1,1,device=x.device) < keep).float() / keep

class _WideBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride, drop=0.3, drop_path=0.1):
        super().__init__()
        self.bn1   = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.drop  = drop; self.dp = DropPath(drop_path)
        self.skip  = None if (in_ch == out_ch and stride == 1) else \
                     nn.Conv2d(in_ch, out_ch, 1, stride, 0, bias=False)
    def forward(self, x):
        h   = F.relu(self.bn1(x))
        out = self.conv1(h)
        out = F.relu(self.bn2(out))
        if self.drop > 0 and self.training:
            out = F.dropout(out, p=self.drop, training=True)
        out = self.dp(self.conv2(out))
        return out + (self.skip(h) if self.skip else x)

class _WideGroup(nn.Module):
    def __init__(self, n, in_ch, out_ch, stride, drop=0.3, drop_path=0.1):
        super().__init__()
        layers = [_WideBlock(in_ch, out_ch, stride, drop, drop_path)]
        for _ in range(n - 1): layers.append(_WideBlock(out_ch, out_ch, 1, drop, drop_path))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class WideResNet(nn.Module):
    def __init__(self, depth=28, width=10, num_classes=10, drop=0.3, drop_path=0.1):
        super().__init__()
        assert (depth - 4) % 6 == 0
        n = (depth - 4) // 6
        chs = [16, 16*width, 32*width, 64*width]
        self.conv0  = nn.Conv2d(3, chs[0], 3, 1, 1, bias=False)
        self.group1 = _WideGroup(n, chs[0], chs[1], 1, drop, drop_path)
        self.group2 = _WideGroup(n, chs[1], chs[2], 2, drop, drop_path)
        self.group3 = _WideGroup(n, chs[2], chs[3], 2, drop, drop_path)
        self.bn     = nn.BatchNorm2d(chs[3])
        self.fc     = nn.Linear(chs[3], num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d): nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear): nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.conv0(x)
        x = self.group1(x); x = self.group2(x); x = self.group3(x)
        x = F.relu(self.bn(x))
        return self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))

def build_model(load_ckpt=None):
    m = WideResNet(28, 10, CFG.num_classes, drop=0.3, drop_path=0.1).to(DEVICE)
    if load_ckpt:
        ckpt_path = Path(load_ckpt)
        if ckpt_path.exists():
            m.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
            print(f"  ✅ Loaded checkpoint: {ckpt_path}")
        else:
            print(f"  ⚠️  WARNING: Checkpoint not found at {ckpt_path}")
            print(f"  ⚠️  Kaggle /kaggle/working/ is session-specific — files from")
            print(f"  ⚠️  previous sessions are gone. Training from scratch instead.")
            print(f"  ⚠️  To use a pretrained model, add the previous notebook as")
            print(f"  ⚠️  a dataset input, then update the path in CFG.pretrained_ckpt")
    if USE_MULTI:
        m = nn.DataParallel(m)
        print(f"  DataParallel across {NUM_GPUS} GPUs")
    print(f"  Params: {sum(p.numel() for p in m.parameters() if p.requires_grad)/1e6:.2f}M")
    return m

# ── 8. Loss ────────────────────────────────────────────────────────────────────
class SoftCE(nn.Module):
    def __init__(self, eps=0.1):
        super().__init__(); self.eps = eps
    def forward(self, logits, targets):
        C = logits.size(1)
        if targets.dim() == 1: targets = F.one_hot(targets, C).float()
        targets = (1 - self.eps) * targets + self.eps / C
        return -(targets * F.log_softmax(logits, 1)).sum(1).mean()

def jsd_loss(logits_clean, logits_aug):
    p_c   = F.softmax(logits_clean.float(), dim=1)
    p_a   = F.softmax(logits_aug.float(), dim=1)
    p_mix = (p_c + p_a) / 2.0
    return (F.kl_div(p_mix.log(), p_c, reduction='batchmean') +
            F.kl_div(p_mix.log(), p_a, reduction='batchmean')) / 2.0

criterion = SoftCE(CFG.label_smooth)

# ── 9. Mixup / CutMix ─────────────────────────────────────────────────────────
def rand_bbox(size, lam):
    H, W = size[-2], size[-1]
    r    = math.sqrt(1 - lam)
    h, w = int(H*r), int(W*r)
    cy, cx = random.randint(0, H), random.randint(0, W)
    return max(0,cx-w//2), max(0,cy-h//2), min(W,cx+w//2), min(H,cy+h//2)

def apply_mix(imgs, lbls):
    perm = torch.randperm(imgs.size(0), device=imgs.device)
    oh   = F.one_hot(lbls, CFG.num_classes).float()
    if random.random() < 0.5:
        lam  = np.random.beta(CFG.mixup_alpha, CFG.mixup_alpha)
        imgs = lam * imgs + (1-lam) * imgs[perm]
    else:
        lam = np.random.beta(CFG.cutmix_alpha, CFG.cutmix_alpha)
        x1,y1,x2,y2 = rand_bbox(imgs.shape, lam)
        imgs = imgs.clone()
        imgs[:,:,y1:y2,x1:x2] = imgs[perm,:,y1:y2,x1:x2]
        lam = 1 - (x2-x1)*(y2-y1)/(imgs.shape[-1]*imgs.shape[-2])
    return imgs, lam*oh + (1-lam)*oh[perm]

def make_sampler(label_list):
    counts  = np.bincount(label_list, minlength=CFG.num_classes).astype(float)
    weights = 1.0 / (counts + 1e-6)
    sw      = torch.tensor([weights[l] for l in label_list], dtype=torch.float)
    return WeightedRandomSampler(sw, len(sw), replacement=True)

# ── 10. Pseudo-labeling (NEW) ─────────────────────────────────────────────────
@torch.no_grad()
def generate_pseudo_labels(model, test_df):
    """
    Run TTA on test set, keep samples where avg confidence > threshold.
    Returns a DataFrame with pseudo-labeled test samples.
    """
    model.eval()
    print(f"\nGenerating pseudo-labels ({CFG.pseudo_tta} TTA passes, conf>{CFG.pseudo_conf}) ...")

    all_probs = []
    for t in range(CFG.pseudo_tta):
        # Use slightly different augmentation each pass
        tta_tfm = transforms.Compose([
            transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),
        ])
        ds = TestDataset(test_df, CFG.test_dir, tta_tfm)
        dl = DataLoader(ds, CFG.batch_size * 2, shuffle=False,
                        num_workers=CFG.num_workers, pin_memory=True)
        probs_t = []
        for imgs, _ in dl:
            with autocast('cuda'):
                logits = model(imgs.to(DEVICE))
            probs_t.append(F.softmax(logits.float(), dim=1).cpu())
        all_probs.append(torch.cat(probs_t, dim=0))
        print(f"  pass {t+1}/{CFG.pseudo_tta}", end='\r')

    avg_probs = torch.stack(all_probs).mean(0)  # (N_test, C)
    confs, preds = avg_probs.max(dim=1)

    # Apply temperature scaling to soften pseudo-labels
    soft_probs = F.softmax(avg_probs / CFG.pseudo_temp, dim=1)

    accepted = confs >= CFG.pseudo_conf
    n_total  = len(test_df)
    n_accept = accepted.sum().item()

    # Fallback: if nothing accepted, lower threshold to grab top 40%
    if n_accept == 0:
        print(f"  ⚠️  0 samples at conf>{CFG.pseudo_conf}, falling back to top-40% by confidence")
        threshold_fallback = confs.kthvalue(int(0.6 * n_total)).values.item()
        accepted = confs >= threshold_fallback
        n_accept = accepted.sum().item()
        print(f"  Fallback threshold: {threshold_fallback:.4f}")

    print(f"\n  Accepted: {n_accept}/{n_total} ({100*n_accept/n_total:.1f}%) test samples")

    # Build pseudo-labeled DataFrame
    pseudo_rows = []
    for i, (id_val, accept, pred, conf) in enumerate(
            zip(test_df['id'], accepted, preds, confs)):
        if accept:
            pseudo_rows.append({
                'id': id_val, 'label': IDX2CLS[pred.item()],
                'label_idx': pred.item(), 'conf': conf.item()
            })

    if len(pseudo_rows) == 0:
        print("  ⚠️  No pseudo-labels generated — skipping pseudo-labeling phase")
        return pd.DataFrame(columns=['id', 'label', 'label_idx', 'conf'])

    pseudo_df = pd.DataFrame(pseudo_rows).reset_index(drop=True)
    pseudo_df.to_csv(CFG.out_dir / 'pseudo_labels.csv', index=False)
    print(f"  Class distribution: {pseudo_df['label'].value_counts().to_dict()}")
    return pseudo_df

# ── 11. Train one epoch ───────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    run_ce = run_jsd = 0.0; nb = 0
    for x_clean, x_aug, lbls in loader:
        x_clean = x_clean.to(DEVICE, non_blocking=True)
        x_aug   = x_aug.to(DEVICE, non_blocking=True)
        lbls    = lbls.to(DEVICE, non_blocking=True)
        B = x_clean.size(0)
        if random.random() < CFG.mix_prob:
            x_clean, soft = apply_mix(x_clean, lbls)
        else:
            soft = lbls
        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda'):
            all_logits   = model(torch.cat([x_clean, x_aug], dim=0))
            logits_clean = all_logits[:B]
            logits_aug   = all_logits[B:]
            loss_ce  = criterion(logits_clean, soft)
            loss_jsd = jsd_loss(logits_clean, logits_aug)
            loss     = loss_ce + CFG.lambda_jsd * loss_jsd
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        params = model.module.parameters() if USE_MULTI else model.parameters()
        nn.utils.clip_grad_norm_(params, CFG.grad_clip)
        scaler.step(optimizer); scaler.update()
        run_ce += loss_ce.item(); run_jsd += loss_jsd.item(); nb += 1
    return run_ce / nb, run_jsd / nb

@torch.no_grad()
def validate(model, loader):
    model.eval(); preds, trues = [], []
    for imgs, lbls in loader:
        with autocast('cuda'):
            p = model(imgs.to(DEVICE)).argmax(1).cpu().numpy()
        preds.extend(p); trues.extend(lbls.numpy())
    return f1_score(trues, preds, average='macro'), np.mean(np.array(preds)==np.array(trues))

@torch.no_grad()
def predict_tta(model, df, img_dir):
    model.eval(); all_probs = []
    for i, tf in enumerate(TTA_TRANSFORMS, 1):
        ds = TestDataset(df, img_dir, tf)
        ld = DataLoader(ds, CFG.batch_size*2, num_workers=CFG.num_workers,
                        shuffle=False, pin_memory=True)
        probs = []
        for imgs, _ in ld:
            with autocast('cuda'):
                logits = model(imgs.to(DEVICE))
            probs.append(F.softmax(logits.float(), 1).cpu().numpy())
        all_probs.append(np.concatenate(probs, 0))
        print(f"    TTA {i}/{CFG.tta_n}", end='\r')
    print()
    return np.mean(all_probs, 0)

# ── 12. Data setup ────────────────────────────────────────────────────────────
print('Loading data ...')
train_df = pd.read_csv(CFG.train_csv)
train_df.columns = [c.strip().lower() for c in train_df.columns]
if   'label' in train_df.columns: train_df['label_idx'] = train_df['label'].map(CLS2IDX)
elif 'class' in train_df.columns: train_df['label_idx'] = train_df['class'].map(CLS2IDX)

test_df = pd.read_csv(CFG.sample_sub)
test_df.columns = [c.strip().lower() for c in test_df.columns]

tr_df, va_df = train_test_split(train_df, test_size=CFG.val_frac,
                                 stratify=train_df['label_idx'], random_state=CFG.seed)
tr_df = tr_df.reset_index(drop=True); va_df = va_df.reset_index(drop=True)
print(f"Train: {len(tr_df):,}  Val: {len(va_df):,}  Test: {len(test_df):,}\n")

va_ds = ValDataset(va_df, CFG.train_dir)
va_ld = DataLoader(va_ds, CFG.batch_size*4, shuffle=False,
                   num_workers=CFG.num_workers, pin_memory=True)

def make_train_loader(df, img_dir, pseudo_df=None, pseudo_dir=None):
    if pseudo_df is not None and len(pseudo_df) > 0:
        ds = CombinedJSDDataset(df, img_dir, pseudo_df, pseudo_dir)
        labels = list(df['label_idx']) + list(pseudo_df['label_idx'])
    else:
        ds = JSDDataset(df, img_dir, use_corruption=True)
        labels = list(df['label_idx'])
    sampler = make_sampler(labels)
    return DataLoader(ds, CFG.batch_size, sampler=sampler,
                      num_workers=CFG.num_workers, pin_memory=True, drop_last=True)

# ── 13. Model / Optimizer / Scheduler ────────────────────────────────────────
print('Building model ...')
model = build_model(CFG.pretrained_ckpt)
raw   = model.module if USE_MULTI else model

optimizer = optim.SGD(raw.parameters(), lr=CFG.lr,
                      momentum=CFG.momentum, weight_decay=CFG.weight_decay,
                      nesterov=CFG.nesterov)
warmup_sched = optim.lr_scheduler.LambdaLR(
    optimizer, lambda ep: (ep+1)/CFG.warmup_eps if ep < CFG.warmup_eps else 1.0)
cosine_sched = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=CFG.T0, T_mult=CFG.T_mult, eta_min=CFG.eta_min)

scaler    = GradScaler('cuda')
swa_model = AveragedModel(raw)
swa_sched = SWALR(optimizer, swa_lr=CFG.swa_lr, anneal_epochs=10)

# ── 14. Training Loop ─────────────────────────────────────────────────────────
best_f1   = 0.0
best_ckpt = CFG.out_dir / 'best_model_v3.pth'
log_rows  = []
t_total   = time.time()
pseudo_df = None   # populated at pseudo_epoch
tr_ld     = make_train_loader(tr_df, CFG.train_dir)  # Phase 1 loader

print(f"\n{'Ep':>4} {'Phase':>7} {'CE':>7} {'JSD':>7} {'F1':>7} {'Acc':>7} {'LR':>9} {'Time':>6}")
print('─' * 65)

for epoch in range(1, CFG.epochs + 1):
    t0 = time.time()

    # ── pseudo-labeling at CFG.pseudo_epoch ──
    if epoch == CFG.pseudo_epoch + 1:
        pseudo_df = generate_pseudo_labels(model, test_df)
        if len(pseudo_df) > 0:
            tr_ld = make_train_loader(tr_df, CFG.train_dir, pseudo_df, CFG.test_dir)
            print(f"  >> Phase 2 loader: {len(tr_ld.dataset):,} samples (original + pseudo)")
        else:
            print("  >> No pseudo-labels — continuing with original training data")

    phase = "PSEUDO" if (pseudo_df is not None and epoch > CFG.pseudo_epoch) else "ORIG"
    loss_ce, loss_jsd = train_one_epoch(model, tr_ld, optimizer, scaler)

    if epoch <= CFG.warmup_eps:
        warmup_sched.step()
    elif epoch < CFG.swa_start:
        cosine_sched.step(epoch - CFG.warmup_eps)
    else:
        swa_model.update_parameters(raw)
        swa_sched.step()

    val_f1, val_acc = validate(model, va_ld)
    lr_now = optimizer.param_groups[0]['lr']

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(raw.state_dict(), best_ckpt)

    log_rows.append({'epoch': epoch, 'phase': phase, 'loss_ce': loss_ce,
                     'loss_jsd': loss_jsd, 'val_f1': val_f1, 'val_acc': val_acc, 'lr': lr_now})

    if epoch % 10 == 0 or epoch <= 5 or epoch == CFG.pseudo_epoch:
        star = ' ★' if val_f1 >= best_f1 else ''
        dt   = time.time() - t0
        print(f'{epoch:>4} {phase:>7} {loss_ce:>7.4f} {loss_jsd:>7.4f} {val_f1:>7.4f} '
              f'{val_acc*100:>6.2f}% {lr_now:>9.6f} {dt:>5.0f}s{star}')

elapsed = (time.time() - t_total) / 3600
print(f'\nTraining done in {elapsed:.2f} hrs  |  Best val F1: {best_f1:.4f}')

# ── 15. SWA finalization ──────────────────────────────────────────────────────
print('\nUpdating SWA BN stats ...')
class _SWAWrap:
    def __init__(self, ldr): self.ldr = ldr
    def __iter__(self):
        for x_c, x_a, y in self.ldr: yield x_c, y

update_bn(_SWAWrap(tr_ld), swa_model, device=DEVICE)
swa_f1, swa_acc = validate(swa_model, va_ld)
print(f'SWA F1: {swa_f1:.4f} | Best epoch F1: {best_f1:.4f}')

if swa_f1 >= best_f1:
    infer_model = swa_model; print('Using SWA model.')
else:
    raw.load_state_dict(torch.load(best_ckpt, map_location=DEVICE, weights_only=True))
    infer_model = raw; print(f'Using best checkpoint (F1={best_f1:.4f}).')

torch.save(raw.state_dict(), CFG.out_dir / 'final_model_v3.pth')
pd.DataFrame(log_rows).to_csv(CFG.out_dir / 'training_log_v3.csv', index=False)

# ── 16. TTA Inference + Submission ───────────────────────────────────────────
print(f'\nRunning TTA inference ({CFG.tta_n} passes) ...')
probs       = predict_tta(infer_model, test_df, CFG.test_dir)
pred_labels = [IDX2CLS[i] for i in probs.argmax(1)]

sub = pd.DataFrame({'id': test_df['id'], 'label': pred_labels})
sub.to_csv(CFG.out_dir / 'submission.csv', index=False)
print(f'\n✅  submission.csv saved  ({len(sub):,} rows)')
print(sub.head(10).to_string(index=False))
print('\nClass distribution:')
print(sub['label'].value_counts().to_string())
print(f'\nTotal time: {(time.time()-t_total)/3600:.2f} hrs')


GPUs: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4

Batch    : 128 (combined→256/step)
Epochs   : 100 | SWA from 60
Pseudo   : epoch 0, conf>0.85, 8 TTA passes
TTA      : 30 passes
Est.time : 217 min (3.6 hrs)

Loading data ...
Train: 27,930  Val: 1,470  Test: 7,600

Building model ...
  ✅ Loaded checkpoint: /kaggle/input/models/kartikeyguptaiitk/trial-6-best/pytorch/default/1/best_model.pth
  DataParallel across 2 GPUs
  Params: 36.48M

  Ep   Phase      CE     JSD      F1     Acc        LR   Time
─────────────────────────────────────────────────────────────────

Generating pseudo-labels (8 TTA passes, conf>0.85) ...
  ⚠️  0 samples at conf>0.85, falling back to top-40% by confidence
  Fallback threshold: 0.4328

  Accepted: 3041/7600 (40.0%) test samples
  Class distribution: {'automobile': 484, 'horse': 474, 'frog': 465, 'ship': 427, 'airplane': 380, 'truck': 288, 'deer': 184, 'dog': 160, 'bird': 100, 'cat': 79}
  >> Phase 2 loader: 30,971 samples (original + pseudo)
   1  PSEUDO  1.3842  

## ShiftGuard10 — v4: Iterative Pseudo-Labeling (Self-Training Iter 2)

_15-pass pseudo-label TTA | 50-pass inference TTA | **Best Val F1: 0.9556**_

---

In [1]:
# =============================================================================
# ShiftGuard10 — best_model_v4.py
# Strategy: Iterative Pseudo-Labeling (Self-Training Iteration 2)
#
# Since v3 achieved 0.893638 (a massive +5% jump), it is now a very strong
# "Teacher" model. For v4, we use the v3 checkpoint to generate a second,
# much cleaner and more confident round of pseudo-labels.
#
# What's new to cross the 0.90 threshold:
#   ✦ Iterative Pseudo-Labeling: Uses your 0.893 LB model as the starting point.
#     The pseudo-labels will be significantly more accurate this time.
#   ✦ Pseudo-label TTA increased from 8 to 15 passes (cleaner labels).
#   ✦ Inference TTA increased from 30 to 50 passes, explicitly including
#     AugMix passes at test time to smooth out extreme shifts.
#   ✦ Safe threshold handling is retained for stability.
# =============================================================================

import os, io, random, time, math, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageEnhance, ImageFilter
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import GradScaler, autocast
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.transforms import RandAugment, RandomErasing
warnings.filterwarnings('ignore')

# ── 0. GPU Setup ──────────────────────────────────────────────────────────────
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_GPUS  = torch.cuda.device_count()
USE_MULTI = NUM_GPUS > 1
print(f"GPUs: {NUM_GPUS}")
for i in range(NUM_GPUS):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
print()

# ── 1. Config ─────────────────────────────────────────────────────────────────
class CFG:
    seed         = 42
    num_classes  = 10
    batch_size   = 128
    num_workers  = 4
    epochs       = 100          # fine-tuning from v3 checkpoint
    warmup_eps   = 2
    swa_start    = 60
    swa_lr       = 0.001

    lr           = 0.005        # lower LR because teacher model is already highly tuned
    momentum     = 0.9
    weight_decay = 5e-4
    nesterov     = True
    grad_clip    = 5.0

    T0           = 25
    T_mult       = 1
    eta_min      = 1e-6
    val_frac     = 0.05

    randaug_n    = 2
    randaug_m    = 12
    mixup_alpha  = 0.4
    cutmix_alpha = 1.0
    mix_prob     = 0.5
    label_smooth = 0.1
    lambda_jsd   = 12.0

    # Pseudo-labeling (Iteration 2)
    pseudo_epoch    = 0         
    pseudo_conf     = 0.85      
    pseudo_tta      = 15        # Increased to 15 passes for even cleaner pseudo-labels
    pseudo_temp     = 2.0

    tta_n        = 50           # Massive 50-pass diverse TTA

    # IMPORTANT: Update this path to the new v3 checkpoint (the one that scored 0.893 LB)
    # E.g., Path('/kaggle/input/your-v3-dataset/final_model_v3.pth')
    pretrained_ckpt = Path('/kaggle/input/models/kartikeyguptaiitk/best-model-v3/pytorch/default/1/best_model_v3.pth')

    if os.path.exists('/kaggle/input'):
        base_dir = Path('/kaggle/input/competitions/shift-guard-10-robust-image-classification-challenge')
        out_dir  = Path('/kaggle/working')
    else:
        base_dir = Path(r'c:\Users\karti\Downloads\shiftgaurd\shift-guard-10-robust-image-classification-challenge')
        out_dir  = Path(r'c:\Users\karti\Downloads\shiftgaurd')

    train_dir  = base_dir / 'train_images'
    test_dir   = base_dir / 'test_images'
    train_csv  = base_dir / 'train_labels.csv'
    sample_sub = base_dir / 'sample_submission.csv'

CLASSES = ['airplane','automobile','bird','cat','deer',
           'dog','frog','horse','ship','truck']
CLS2IDX = {c: i for i, c in enumerate(CLASSES)}
IDX2CLS = {i: c for c, i in CLS2IDX.items()}
MEAN    = [0.4914, 0.4822, 0.4465]
STD     = [0.2470, 0.2435, 0.2616]

print(f"Batch    : {CFG.batch_size} (combined→{CFG.batch_size*2}/step)")
print(f"Epochs   : {CFG.epochs} | SWA from {CFG.swa_start}")
print(f"Pseudo   : epoch {CFG.pseudo_epoch}, conf>{CFG.pseudo_conf}, {CFG.pseudo_tta} TTA passes")
print(f"TTA      : {CFG.tta_n} passes")
est = CFG.epochs * 130 / 60 if USE_MULTI else CFG.epochs * 270 / 60
print(f"Est.time : {est:.0f} min ({est/60:.1f} hrs)\n")

# ── 2. Reproducibility ────────────────────────────────────────────────────────
def seed_everything(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    os.environ['PYTHONHASHSEED'] = str(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

seed_everything(CFG.seed)

# ── 3. Corruption augmentations ───────────────────────────────────────────────
def gaussian_noise(img, std_range=(0.02, 0.12)):
    arr = np.array(img, dtype=np.float32) / 255.
    std = random.uniform(*std_range)
    arr = np.clip(arr + np.random.randn(*arr.shape) * std, 0, 1)
    return Image.fromarray((arr * 255).astype(np.uint8))

def motion_blur(img, max_size=5):
    size = random.choice([3, 5])
    return img.filter(ImageFilter.BoxBlur(size // 2))

def pixelate(img, factor=None):
    w, h = img.size
    f = factor or random.randint(2, 4)
    small = img.resize((max(1, w // f), max(1, h // f)), Image.BILINEAR)
    return small.resize((w, h), Image.NEAREST)

def jpeg_compress(img, quality_range=(20, 60)):
    quality = random.randint(*quality_range)
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=quality)
    buf.seek(0)
    return Image.open(buf).convert('RGB')

CORRUPTION_OPS = [gaussian_noise, motion_blur, pixelate, jpeg_compress]

def apply_corruption(img):
    if random.random() < 0.3:
        return random.choice(CORRUPTION_OPS)(img)
    return img

# ── 4. AugMix ─────────────────────────────────────────────────────────────────
def _int(level, maxval): return int(level * maxval / 10)
def _flt(level, maxval): return float(level) * maxval / 10.

AUG_OPS = [
    lambda img, l: ImageOps.autocontrast(img),
    lambda img, l: ImageOps.equalize(img),
    lambda img, l: ImageOps.posterize(img, max(1, 4 - _int(l, 4))),
    lambda img, l: ImageOps.solarize(img, 256 - _int(l, 256)),
    lambda img, l: img.rotate(_int(l, 30) * random.choice([-1, 1]), fillcolor=(128,128,128)),
    lambda img, l: img.transform(img.size, Image.AFFINE, (1, _flt(l,.3)*random.choice([-1,1]), 0, 0, 1, 0), fillcolor=(128,128,128)),
    lambda img, l: img.transform(img.size, Image.AFFINE, (1, 0, 0, _flt(l,.3)*random.choice([-1,1]), 1, 0), fillcolor=(128,128,128)),
    lambda img, l: ImageEnhance.Color(img).enhance(_flt(l, 1.8) + 0.1),
    lambda img, l: ImageEnhance.Contrast(img).enhance(_flt(l, 1.8) + 0.1),
    lambda img, l: ImageEnhance.Brightness(img).enhance(_flt(l, 1.8) + 0.1),
    lambda img, l: ImageEnhance.Sharpness(img).enhance(_flt(l, 1.8) + 0.1),
    lambda img, l: gaussian_noise(img),
    lambda img, l: motion_blur(img),
    lambda img, l: pixelate(img),
]

def augmix_pil(image, severity=3, width=3, alpha=1.):
    ws  = np.float32(np.random.dirichlet([alpha] * width))
    m   = np.float32(np.random.beta(alpha, alpha))
    mix = np.zeros_like(np.array(image), dtype=np.float32)
    for i in range(width):
        img_aug = image.copy()
        for _ in range(np.random.randint(1, 4)):
            img_aug = random.choice(AUG_OPS)(img_aug, np.random.randint(1, severity + 1))
        mix += ws[i] * np.array(img_aug, dtype=np.float32)
    return Image.fromarray(np.uint8(np.clip((1-m)*np.array(image, np.float32) + m*mix, 0, 255)))

# ── 5. Transforms ─────────────────────────────────────────────────────────────
clean_tfm = transforms.Compose([
    transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

augmix_base_tfm = transforms.Compose([
    transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
    transforms.RandomHorizontalFlip(),
    RandAugment(num_ops=CFG.randaug_n, magnitude=CFG.randaug_m),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    RandomErasing(p=0.4, scale=(0.02, 0.2), ratio=(0.3, 3.3), value='random'),
])

val_tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

def build_tta_transforms():
    """50 TTA passes: robust combination for LB push."""
    tfms    = []
    to_norm = [transforms.ToTensor(), transforms.Normalize(MEAN, STD)]
    
    # 20 crop/flip passes (base)
    offsets = [(0,0),(0,8),(8,0),(8,8),(4,4),(2,6),(6,2),(0,4),(4,0),(4,8)]
    for flip in [False, True]:
        for r, c in offsets:
            ops = []
            if flip: ops.append(transforms.RandomHorizontalFlip(p=1.0))
            ops.append(transforms.Pad(4, padding_mode='reflect'))
            ops.append(transforms.Lambda(lambda img,r=r,c=c: transforms.functional.crop(img,r,c,32,32)))
            ops += to_norm
            tfms.append(transforms.Compose(ops))

    # 10 passes: light color jitter
    jitter = transforms.ColorJitter(0.15, 0.15, 0.15)
    for flip in [False, True]:
        for r, c in offsets[:5]:
            ops = []
            if flip: ops.append(transforms.RandomHorizontalFlip(p=1.0))
            ops += [jitter, transforms.Pad(4, padding_mode='reflect')]
            ops.append(transforms.Lambda(lambda img,r=r,c=c: transforms.functional.crop(img,r,c,32,32)))
            ops += to_norm
            tfms.append(transforms.Compose(ops))

    # 10 passes: blur & noise combined randomly
    for flip in [False, True]:
        for r, c in offsets[:5]:
            ops = []
            if flip: ops.append(transforms.RandomHorizontalFlip(p=1.0))
            def b_n(img):
                if random.random() < 0.5: img = img.filter(ImageFilter.BoxBlur(1))
                if random.random() < 0.5: img = gaussian_noise(img, (0.01, 0.04))
                return img
            ops.append(transforms.Lambda(b_n))
            ops.append(transforms.Pad(4, padding_mode='reflect'))
            ops.append(transforms.Lambda(lambda img,r=r,c=c: transforms.functional.crop(img,r,c,32,32)))
            ops += to_norm
            tfms.append(transforms.Compose(ops))

    # 10 passes: AugMix on inference to combat distribution shift directly
    for _ in range(10):
        ops = [
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.Lambda(lambda img: augmix_pil(img, severity=2, width=2)),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD)
        ]
        tfms.append(transforms.Compose(ops))

    return tfms[:CFG.tta_n]

TTA_TRANSFORMS = build_tta_transforms()

# ── 6. Datasets ────────────────────────────────────────────────────────────────
def _resolve_path(img_dir, id_):
    p = img_dir / f"{str(id_).zfill(6)}.png"
    if not p.exists():
        p = img_dir / f"{id_}.png"
    return p

class JSDDataset(Dataset):
    def __init__(self, df, img_dir, use_corruption=True):
        self.df   = df.reset_index(drop=True)
        self.dir  = img_dir
        self.use_c = use_corruption
        self.paths = [_resolve_path(img_dir, r['id']) for _, r in df.iterrows()]
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.use_c:
            img = apply_corruption(img)
        x_clean = clean_tfm(img)
        x_aug   = augmix_base_tfm(augmix_pil(img))
        return x_clean, x_aug, int(self.df.iloc[idx]['label_idx'])

class CombinedJSDDataset(Dataset):
    def __init__(self, orig_df, orig_dir, pseudo_df, pseudo_dir):
        self.items = []
        for _, row in orig_df.iterrows():
            self.items.append((_resolve_path(orig_dir, row['id']), int(row['label_idx'])))
        for _, row in pseudo_df.iterrows():
            self.items.append((_resolve_path(pseudo_dir, row['id']), int(row['label_idx'])))

    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        path, label = self.items[idx]
        img = Image.open(path).convert('RGB')
        img = apply_corruption(img)
        x_clean = clean_tfm(img)
        x_aug   = augmix_base_tfm(augmix_pil(img))
        return x_clean, x_aug, label

class ValDataset(Dataset):
    def __init__(self, df, img_dir):
        self.df    = df.reset_index(drop=True)
        self.paths = [_resolve_path(img_dir, r['id']) for _, r in df.iterrows()]
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        return val_tfm(Image.open(self.paths[idx]).convert('RGB')), int(self.df.iloc[idx]['label_idx'])

class TestDataset(Dataset):
    def __init__(self, df, img_dir, tfm=None):
        self.df    = df.reset_index(drop=True)
        self.tfm   = tfm
        self.paths = [_resolve_path(img_dir, r['id']) for _, r in df.iterrows()]
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.tfm: img = self.tfm(img)
        return img, str(self.df.iloc[idx]['id'])

# ── 7. Model: WideResNet-28-10 + DropPath ─────────────────────────────────────
class DropPath(nn.Module):
    def __init__(self, p=0.1):
        super().__init__(); self.p = p
    def forward(self, x):
        if not self.training or self.p == 0: return x
        keep = 1 - self.p
        return x * (torch.rand(x.shape[0],1,1,1,device=x.device) < keep).float() / keep

class _WideBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride, drop=0.3, drop_path=0.1):
        super().__init__()
        self.bn1   = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.drop  = drop; self.dp = DropPath(drop_path)
        self.skip  = None if (in_ch == out_ch and stride == 1) else \
                     nn.Conv2d(in_ch, out_ch, 1, stride, 0, bias=False)
    def forward(self, x):
        h   = F.relu(self.bn1(x))
        out = self.conv1(h)
        out = F.relu(self.bn2(out))
        if self.drop > 0 and self.training:
            out = F.dropout(out, p=self.drop, training=True)
        out = self.dp(self.conv2(out))
        return out + (self.skip(h) if self.skip else x)

class _WideGroup(nn.Module):
    def __init__(self, n, in_ch, out_ch, stride, drop=0.3, drop_path=0.1):
        super().__init__()
        layers = [_WideBlock(in_ch, out_ch, stride, drop, drop_path)]
        for _ in range(n - 1): layers.append(_WideBlock(out_ch, out_ch, 1, drop, drop_path))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class WideResNet(nn.Module):
    def __init__(self, depth=28, width=10, num_classes=10, drop=0.3, drop_path=0.1):
        super().__init__()
        assert (depth - 4) % 6 == 0
        n = (depth - 4) // 6
        chs = [16, 16*width, 32*width, 64*width]
        self.conv0  = nn.Conv2d(3, chs[0], 3, 1, 1, bias=False)
        self.group1 = _WideGroup(n, chs[0], chs[1], 1, drop, drop_path)
        self.group2 = _WideGroup(n, chs[1], chs[2], 2, drop, drop_path)
        self.group3 = _WideGroup(n, chs[2], chs[3], 2, drop, drop_path)
        self.bn     = nn.BatchNorm2d(chs[3])
        self.fc     = nn.Linear(chs[3], num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d): nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear): nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.conv0(x)
        x = self.group1(x); x = self.group2(x); x = self.group3(x)
        x = F.relu(self.bn(x))
        return self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))

def build_model(load_ckpt=None):
    m = WideResNet(28, 10, CFG.num_classes, drop=0.3, drop_path=0.1).to(DEVICE)
    if load_ckpt:
        ckpt_path = Path(load_ckpt)
        if ckpt_path.exists():
            m.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
            print(f"  ✅ Loaded checkpoint: {ckpt_path}")
        else:
            print(f"  ⚠️  WARNING: Checkpoint not found at {ckpt_path}")
            print(f"  ⚠️  Kaggle /kaggle/working/ is session-specific — files from")
            print(f"  ⚠️  previous sessions are gone. Training from scratch instead.")
            print(f"  ⚠️  To use a pretrained model, add the previous notebook as")
            print(f"  ⚠️  a dataset input, then update the path in CFG.pretrained_ckpt")
    if USE_MULTI:
        m = nn.DataParallel(m)
        print(f"  DataParallel across {NUM_GPUS} GPUs")
    print(f"  Params: {sum(p.numel() for p in m.parameters() if p.requires_grad)/1e6:.2f}M")
    return m

# ── 8. Loss ────────────────────────────────────────────────────────────────────
class SoftCE(nn.Module):
    def __init__(self, eps=0.1):
        super().__init__(); self.eps = eps
    def forward(self, logits, targets):
        C = logits.size(1)
        if targets.dim() == 1: targets = F.one_hot(targets, C).float()
        targets = (1 - self.eps) * targets + self.eps / C
        return -(targets * F.log_softmax(logits, 1)).sum(1).mean()

def jsd_loss(logits_clean, logits_aug):
    p_c   = F.softmax(logits_clean.float(), dim=1)
    p_a   = F.softmax(logits_aug.float(), dim=1)
    p_mix = (p_c + p_a) / 2.0
    return (F.kl_div(p_mix.log(), p_c, reduction='batchmean') +
            F.kl_div(p_mix.log(), p_a, reduction='batchmean')) / 2.0

criterion = SoftCE(CFG.label_smooth)

# ── 9. Mixup / CutMix ─────────────────────────────────────────────────────────
def rand_bbox(size, lam):
    H, W = size[-2], size[-1]
    r    = math.sqrt(1 - lam)
    h, w = int(H*r), int(W*r)
    cy, cx = random.randint(0, H), random.randint(0, W)
    return max(0,cx-w//2), max(0,cy-h//2), min(W,cx+w//2), min(H,cy+h//2)

def apply_mix(imgs, lbls):
    perm = torch.randperm(imgs.size(0), device=imgs.device)
    oh   = F.one_hot(lbls, CFG.num_classes).float()
    if random.random() < 0.5:
        lam  = np.random.beta(CFG.mixup_alpha, CFG.mixup_alpha)
        imgs = lam * imgs + (1-lam) * imgs[perm]
    else:
        lam = np.random.beta(CFG.cutmix_alpha, CFG.cutmix_alpha)
        x1,y1,x2,y2 = rand_bbox(imgs.shape, lam)
        imgs = imgs.clone()
        imgs[:,:,y1:y2,x1:x2] = imgs[perm,:,y1:y2,x1:x2]
        lam = 1 - (x2-x1)*(y2-y1)/(imgs.shape[-1]*imgs.shape[-2])
    return imgs, lam*oh + (1-lam)*oh[perm]

def make_sampler(label_list):
    counts  = np.bincount(label_list, minlength=CFG.num_classes).astype(float)
    weights = 1.0 / (counts + 1e-6)
    sw      = torch.tensor([weights[l] for l in label_list], dtype=torch.float)
    return WeightedRandomSampler(sw, len(sw), replacement=True)

# ── 10. Pseudo-labeling ───────────────────────────────────────────────────────
@torch.no_grad()
def generate_pseudo_labels(model, test_df):
    model.eval()
    print(f"\nGenerating pseudo-labels ({CFG.pseudo_tta} TTA passes, conf>{CFG.pseudo_conf}) ...")

    all_probs = []
    for t in range(CFG.pseudo_tta):
        tta_tfm = transforms.Compose([
            transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
            transforms.RandomHorizontalFlip(),
            transforms.Lambda(lambda img: apply_corruption(img) if random.random() < 0.2 else img),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),
        ])
        ds = TestDataset(test_df, CFG.test_dir, tta_tfm)
        dl = DataLoader(ds, CFG.batch_size * 2, shuffle=False,
                        num_workers=CFG.num_workers, pin_memory=True)
        probs_t = []
        for imgs, _ in dl:
            with autocast('cuda'):
                logits = model(imgs.to(DEVICE))
            probs_t.append(F.softmax(logits.float(), dim=1).cpu())
        all_probs.append(torch.cat(probs_t, dim=0))
        print(f"  pass {t+1}/{CFG.pseudo_tta}", end='\r')

    avg_probs = torch.stack(all_probs).mean(0)
    confs, preds = avg_probs.max(dim=1)

    accepted = confs >= CFG.pseudo_conf
    n_total  = len(test_df)
    n_accept = accepted.sum().item()

    if n_accept == 0:
        print(f"  ⚠️  0 samples at conf>{CFG.pseudo_conf}, falling back to top-40% by confidence")
        threshold_fallback = confs.kthvalue(int(0.6 * n_total)).values.item()
        accepted = confs >= threshold_fallback
        n_accept = accepted.sum().item()
        print(f"  Fallback threshold: {threshold_fallback:.4f}")

    print(f"\n  Accepted: {n_accept}/{n_total} ({100*n_accept/n_total:.1f}%) test samples")

    pseudo_rows = []
    for i, (id_val, accept, pred, conf) in enumerate(
            zip(test_df['id'], accepted, preds, confs)):
        if accept:
            pseudo_rows.append({
                'id': id_val, 'label': IDX2CLS[pred.item()],
                'label_idx': pred.item(), 'conf': conf.item()
            })

    if len(pseudo_rows) == 0:
        print("  ⚠️  No pseudo-labels generated — skipping pseudo-labeling phase")
        return pd.DataFrame(columns=['id', 'label', 'label_idx', 'conf'])

    pseudo_df = pd.DataFrame(pseudo_rows).reset_index(drop=True)
    pseudo_df.to_csv(CFG.out_dir / 'pseudo_labels_iter2.csv', index=False)
    print(f"  Class distribution: {pseudo_df['label'].value_counts().to_dict()}")
    return pseudo_df

# ── 11. Train one epoch ───────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    run_ce = run_jsd = 0.0; nb = 0
    for x_clean, x_aug, lbls in loader:
        x_clean = x_clean.to(DEVICE, non_blocking=True)
        x_aug   = x_aug.to(DEVICE, non_blocking=True)
        lbls    = lbls.to(DEVICE, non_blocking=True)
        B = x_clean.size(0)
        if random.random() < CFG.mix_prob:
            x_clean, soft = apply_mix(x_clean, lbls)
        else:
            soft = lbls
        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda'):
            all_logits   = model(torch.cat([x_clean, x_aug], dim=0))
            logits_clean = all_logits[:B]
            logits_aug   = all_logits[B:]
            loss_ce  = criterion(logits_clean, soft)
            loss_jsd = jsd_loss(logits_clean, logits_aug)
            loss     = loss_ce + CFG.lambda_jsd * loss_jsd
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        params = model.module.parameters() if USE_MULTI else model.parameters()
        nn.utils.clip_grad_norm_(params, CFG.grad_clip)
        scaler.step(optimizer); scaler.update()
        run_ce += loss_ce.item(); run_jsd += loss_jsd.item(); nb += 1
    return run_ce / nb, run_jsd / nb

@torch.no_grad()
def validate(model, loader):
    model.eval(); preds, trues = [], []
    for imgs, lbls in loader:
        with autocast('cuda'):
            p = model(imgs.to(DEVICE)).argmax(1).cpu().numpy()
        preds.extend(p); trues.extend(lbls.numpy())
    return f1_score(trues, preds, average='macro'), np.mean(np.array(preds)==np.array(trues))

@torch.no_grad()
def predict_tta(model, df, img_dir):
    model.eval(); all_probs = []
    for i, tf in enumerate(TTA_TRANSFORMS, 1):
        ds = TestDataset(df, img_dir, tf)
        ld = DataLoader(ds, CFG.batch_size*2, num_workers=CFG.num_workers,
                        shuffle=False, pin_memory=True)
        probs = []
        for imgs, _ in ld:
            with autocast('cuda'):
                logits = model(imgs.to(DEVICE))
            probs.append(F.softmax(logits.float(), 1).cpu().numpy())
        all_probs.append(np.concatenate(probs, 0))
        print(f"    TTA {i}/{CFG.tta_n}", end='\r')
    print()
    return np.mean(all_probs, 0)

# ── 12. Data setup ────────────────────────────────────────────────────────────
print('Loading data ...')
train_df = pd.read_csv(CFG.train_csv)
train_df.columns = [c.strip().lower() for c in train_df.columns]
if   'label' in train_df.columns: train_df['label_idx'] = train_df['label'].map(CLS2IDX)
elif 'class' in train_df.columns: train_df['label_idx'] = train_df['class'].map(CLS2IDX)

test_df = pd.read_csv(CFG.sample_sub)
test_df.columns = [c.strip().lower() for c in test_df.columns]

tr_df, va_df = train_test_split(train_df, test_size=CFG.val_frac,
                                 stratify=train_df['label_idx'], random_state=CFG.seed)
tr_df = tr_df.reset_index(drop=True); va_df = va_df.reset_index(drop=True)
print(f"Train: {len(tr_df):,}  Val: {len(va_df):,}  Test: {len(test_df):,}\n")

va_ds = ValDataset(va_df, CFG.train_dir)
va_ld = DataLoader(va_ds, CFG.batch_size*4, shuffle=False,
                   num_workers=CFG.num_workers, pin_memory=True)

def make_train_loader(df, img_dir, pseudo_df=None, pseudo_dir=None):
    if pseudo_df is not None and len(pseudo_df) > 0:
        ds = CombinedJSDDataset(df, img_dir, pseudo_df, pseudo_dir)
        labels = list(df['label_idx']) + list(pseudo_df['label_idx'])
    else:
        ds = JSDDataset(df, img_dir, use_corruption=True)
        labels = list(df['label_idx'])
    sampler = make_sampler(labels)
    return DataLoader(ds, CFG.batch_size, sampler=sampler,
                      num_workers=CFG.num_workers, pin_memory=True, drop_last=True)

# ── 13. Model / Optimizer / Scheduler ────────────────────────────────────────
print('Building model ...')
model = build_model(CFG.pretrained_ckpt)
raw   = model.module if USE_MULTI else model

optimizer = optim.SGD(raw.parameters(), lr=CFG.lr,
                      momentum=CFG.momentum, weight_decay=CFG.weight_decay,
                      nesterov=CFG.nesterov)
warmup_sched = optim.lr_scheduler.LambdaLR(
    optimizer, lambda ep: (ep+1)/CFG.warmup_eps if ep < CFG.warmup_eps else 1.0)
cosine_sched = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=CFG.T0, T_mult=CFG.T_mult, eta_min=CFG.eta_min)

scaler    = GradScaler('cuda')
swa_model = AveragedModel(raw)
swa_sched = SWALR(optimizer, swa_lr=CFG.swa_lr, anneal_epochs=10)

# ── 14. Training Loop ─────────────────────────────────────────────────────────
best_f1   = 0.0
best_ckpt = CFG.out_dir / 'best_model_v4.pth'
log_rows  = []
t_total   = time.time()
pseudo_df = None
tr_ld     = make_train_loader(tr_df, CFG.train_dir)

print(f"\n{'Ep':>4} {'Phase':>7} {'CE':>7} {'JSD':>7} {'F1':>7} {'Acc':>7} {'LR':>9} {'Time':>6}")
print('─' * 65)

for epoch in range(1, CFG.epochs + 1):
    t0 = time.time()

    if epoch == CFG.pseudo_epoch + 1:
        pseudo_df = generate_pseudo_labels(model, test_df)
        if len(pseudo_df) > 0:
            tr_ld = make_train_loader(tr_df, CFG.train_dir, pseudo_df, CFG.test_dir)
            print(f"  >> Phase 2 loader: {len(tr_ld.dataset):,} samples (original + pseudo)")
        else:
            print("  >> No pseudo-labels — continuing with original training data")

    phase = "PSEUDO" if (pseudo_df is not None and epoch > CFG.pseudo_epoch) else "ORIG"
    loss_ce, loss_jsd = train_one_epoch(model, tr_ld, optimizer, scaler)

    if epoch <= CFG.warmup_eps:
        warmup_sched.step()
    elif epoch < CFG.swa_start:
        cosine_sched.step(epoch - CFG.warmup_eps)
    else:
        swa_model.update_parameters(raw)
        swa_sched.step()

    val_f1, val_acc = validate(model, va_ld)
    lr_now = optimizer.param_groups[0]['lr']

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(raw.state_dict(), best_ckpt)

    log_rows.append({'epoch': epoch, 'phase': phase, 'loss_ce': loss_ce,
                     'loss_jsd': loss_jsd, 'val_f1': val_f1, 'val_acc': val_acc, 'lr': lr_now})

    if epoch % 10 == 0 or epoch <= 5 or epoch == CFG.pseudo_epoch:
        star = ' ★' if val_f1 >= best_f1 else ''
        dt   = time.time() - t0
        print(f'{epoch:>4} {phase:>7} {loss_ce:>7.4f} {loss_jsd:>7.4f} {val_f1:>7.4f} '
              f'{val_acc*100:>6.2f}% {lr_now:>9.6f} {dt:>5.0f}s{star}')

elapsed = (time.time() - t_total) / 3600
print(f'\nTraining done in {elapsed:.2f} hrs  |  Best val F1: {best_f1:.4f}')

# ── 15. SWA finalization ──────────────────────────────────────────────────────
print('\nUpdating SWA BN stats ...')
class _SWAWrap:
    def __init__(self, ldr): self.ldr = ldr
    def __iter__(self):
        for x_c, x_a, y in self.ldr: yield x_c, y

update_bn(_SWAWrap(tr_ld), swa_model, device=DEVICE)
swa_f1, swa_acc = validate(swa_model, va_ld)
print(f'SWA F1: {swa_f1:.4f} | Best epoch F1: {best_f1:.4f}')

if swa_f1 >= best_f1:
    infer_model = swa_model; print('Using SWA model.')
else:
    raw.load_state_dict(torch.load(best_ckpt, map_location=DEVICE, weights_only=True))
    infer_model = raw; print(f'Using best checkpoint (F1={best_f1:.4f}).')

torch.save(raw.state_dict(), CFG.out_dir / 'final_model_v4.pth')
pd.DataFrame(log_rows).to_csv(CFG.out_dir / 'training_log_v4.csv', index=False)

# ── 16. TTA Inference + Submission ───────────────────────────────────────────
print(f'\nRunning TTA inference ({CFG.tta_n} passes) ...')
probs       = predict_tta(infer_model, test_df, CFG.test_dir)
pred_labels = [IDX2CLS[i] for i in probs.argmax(1)]

sub = pd.DataFrame({'id': test_df['id'], 'label': pred_labels})
sub.to_csv(CFG.out_dir / 'submission.csv', index=False)
print(f'\n✅  submission.csv saved  ({len(sub):,} rows)')
print(sub.head(10).to_string(index=False))
print('\nClass distribution:')
print(sub['label'].value_counts().to_string())
print(f'\nTotal time: {(time.time()-t_total)/3600:.2f} hrs')


GPUs: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4

Batch    : 128 (combined→256/step)
Epochs   : 100 | SWA from 60
Pseudo   : epoch 0, conf>0.85, 15 TTA passes
TTA      : 50 passes
Est.time : 217 min (3.6 hrs)

Loading data ...
Train: 27,930  Val: 1,470  Test: 7,600

Building model ...
  ✅ Loaded checkpoint: /kaggle/input/models/kartikeyguptaiitk/best-model-v3/pytorch/default/1/best_model_v3.pth
  DataParallel across 2 GPUs
  Params: 36.48M

  Ep   Phase      CE     JSD      F1     Acc        LR   Time
─────────────────────────────────────────────────────────────────

Generating pseudo-labels (15 TTA passes, conf>0.85) ...
  ⚠️  0 samples at conf>0.85, falling back to top-40% by confidence
  Fallback threshold: 0.4837

  Accepted: 3041/7600 (40.0%) test samples
  Class distribution: {'ship': 589, 'frog': 504, 'truck': 480, 'horse': 451, 'automobile': 366, 'deer': 199, 'airplane': 178, 'dog': 134, 'bird': 77, 'cat': 63}
  >> Phase 2 loader: 30,971 samples (original + pseudo)
   1  PSEUDO  1.2